In [ ]:
import os
os.environ["POLARS_FORCE_NEW_STREAMING"]="1"
import datetime
import colormaps
from cartopy import crs as ccrs
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
import PIL

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
    to_expr,
    get_index_columns,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    connected_from_cross,
    slowness_expr,
    spells_from_cross_catd_simple,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
    props_vs_quantiles,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import (
    create_bootstrapped_times,
    odds_ratio,
    is_signif_OR,
    common_OR,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}
# pl.Config.set_verbose(True)
# pl.Config.set_streaming_chunk_size(500)  
basepath = Path(f"{DATADIR}/exp9")

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
vsq = jets.group_by("time", "jet ID").agg(vsq=0.5 * (pl.col("v") ** 2).mean())
props = pl.read_parquet(basepath.joinpath("props.parquet"))
props = props.join(vsq, on=["time", "jet ID"])
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(
    pers=pl.col("slowness").sum()
)
props_catd = props_catd.join(pers, on=("time", "jet"))
props_catd_summer = summer_filter.join(props_catd, on="time")

dist_threshold = 2.0e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
spells_list = spells_from_cross(
    jets,
    cross,
    None,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
    n_STJ=30,
    n_EDJ=30,
    season=summer,
    STJ_lat_threshold=30,
)
_, summary_comp = connected_from_cross(
    jets,
    cross,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
)
jet_column = (
    pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
    .then(pl.lit("EDJ"))
    .otherwise(pl.lit("STJ"))
)
summary_comp = (
    summary_comp.filter(pl.col("len") > 1)
    .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
    .join(props["time", "jet ID", "is_polar"], on=("time", "jet ID"))
    .with_columns(
        jet=jet_column,
        slowness=slowness_expr(),
    )
    .with_columns(
        slowness_sum=pl.col("slowness").sum().over("spell"),
    )
    # .join(props, on=["time", "jet ID"], suffix="_fromprops")
    .sort("time", "jet ID")
)
summer_summary_comp = summary_comp.filter(
    pl.col("time")
    .is_in(pl.lit(summer.implode().first(), pl.List(pl.Datetime("ms"))))
    .over("spell")
)

for jet_name, spells in spells_list.items():
    print(jet_name, spells["spell"].n_unique())
    print(jet_name, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

# outlier suppression

In [ ]:
def _zoubisou(var, lev, days = None):
    key = f"{var}{lev}"
    if days is not None:
        key2 = f"{key}_{days}days"
    else:
        key2 = key
    var1 = f"{var}_interp"
    var2 = f"{key}_interp"
    f1 = basepath.joinpath(f"{key2}_relative.parquet")
    f2 = basepath.joinpath(f"{key2}_orig_relative.parquet")
    if f2.is_file():
        return
    os.rename(f1, f2)
    df = pl.read_parquet(f2)
    if var1 in df.columns:
        df = df.rename({var1: var2})
    df = df.with_columns(gb=pl.col(var2).abs().log10().round(2))
    filter_ = df.group_by("gb").agg(pl.len()).sort("gb")
    expr = (pl.col("len") > 10000) | (pl.col("gb") < 0)
    expr = expr.rle_id() == expr.rle_id().filter(expr).mode().first()
    filter_ = filter_.with_columns(filter=expr)
    df = df.join(filter_.drop("len"), on="gb").drop("gb")
    if var == "EKE":
        filter_ = pl.col("filter") & (pl.col(var2) > 0)
    else:
        filter_ = pl.col("filter")
    agg = {var2: pl.when(filter_).then(var2).otherwise(None)}
    df = df.with_columns(**agg).drop("filter")
    df.write_parquet(f1)

for var, lev, days in tqdm(product(["EKE" ,"EMFconv"], [320, 330, 340, 350], [10, 20, 30]), total=24):
    _zoubisou(var, lev, days)
    
for var, lev in tqdm(product(["PV"], [320, 330, 340, 350]), total=4):
    _zoubisou(var, lev)
    
for var, lev, days in tqdm(product(["EKE" ,"EMFconv"], [320, 330, 340, 350], [10, 20, 30]), total=24):
    key = f"{var}{lev}"
    key2 = f"{key}_{days}days"
    var1 = f"{var}_interp"
    var2 = f"{key}_interp"
    var3 = f"{key2}_interp"
    f1 = basepath.joinpath(f"{key2}_relative.parquet")
    df = pl.read_parquet(f1)
    df = df.rename({var2: var3})
    df.write_parquet(f1)

# more stuff

In [ ]:
filters_for_variables = {
    "APVOany": ["cold", "warm"],
    "CPVOany": ["cold", "warm"],
    "t2m": ["cold", "warm"],
    "tp": [
        "warm_entrance",
        "cold_exit",
        "cold",
        "warm",
        "warm_far_entrance",
    ],
    "theta": [
        "cold",
        "warm",
    ],
    "theta:grad": [
        "core",
    ],
    "PV330": [
        "cold",
        "warm",
    ],
    # "PV330:grad": [
    #     "core",
    # ],
    "PV350": [
        "cold",
        "warm",
    ],
    # "PV350:grad": [
    #     "core",
    # ],
    # "EKE330_10days": ["cold", "warm", "core"],
    # "EMFconv330_10days": ["cold", "warm", "core"],
    # "EKE340_10days": ["cold", "warm", "core"],
    # "EMFconv340_10days": ["cold", "warm", "core"],
    # "EKE350_10days": ["cold", "warm", "core"],
    # "EMFconv350_10days": ["cold", "warm", "core"],
}
for level in [250, 300, 850]:
    filters_for_variables[f"t{level}"] = ["cold", "warm"]
    filters_for_variables[f"t{level}:grad"] = ["core"]
n_days = 5
for level in [200, 225, 250, 300, 850]:
    for var in ["EKE", "hor", "vert"]:
        filters_for_variables[f"{var}{level}_{n_days}days"] = ["core"]
for level, n_days in product([250, 300], [5, 10, 20, 30]):
    for var in ["EKE"]:
        filters_for_variables[f"{var}{level}_{n_days}days"] = ["core"]
reduce_dict = {
    var: pl.Expr.any for var in ["APVOany", "CPVOany"]
}

In [ ]:
from jetutils.geospatial import prepare_last_step_1, prepare_last_step_2

pwe_path = basepath.joinpath("props_with_extras.parquet")
if not pwe_path.is_file():
    props_with_extras = prepare_last_step_1(
        basepath, filters_for_variables, props, reduce_dict
    )
    props_with_extras.write_parquet(pwe_path)
else:
    props_with_extras = pl.read_parquet(pwe_path)
props_with_extras = props_with_extras.join(vsq, on=["time", "jet ID"])

tss_path = Path(DATADIR, "ERA5/timeseries")
for f in tss_path.glob("*.parquet"):
    ts = pl.read_parquet(f)
    if ts.columns[-1] == "tp_anom":
        ts = ts.with_columns(pl.col("tp_anom") * 1000)
    if ts.columns[-1] == "tp":
        ts = ts.with_columns(pl.col("tp") * 1000)
    if f.stem.split("_")[0] == "region":
        ts = ts.pivot("region", index="time", column_naming="combine")
    else:
        ts = ts.rename({ts.columns[-1]: f.stem})
    props_with_extras = props_with_extras.join(ts, on="time", how="left")
props_with_extras_summer = summer_filter.join(props_with_extras, on="time")
grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
schema = {"year": pl.UInt32(), "month": pl.UInt32(), "day": pl.UInt32(), "x": pl.Float32(), "y": pl.Float32(), "phase": pl.UInt8(), "A": pl.Float32()}
mjo = pl.from_pandas(pd.read_csv(f"{DATADIR}/ERA5/timeseries/mjo.txt", sep=r"\s+", names=list(schema)), schema_overrides=schema)
mjo = ALL_TIMES.join(mjo.select(time=pl.datetime("year", "month", "day", time_unit="ms"), phase="phase"), on="time", how="left").with_columns(pl.col("phase").fill_null(strategy="forward"))

props_with_extras = props_with_extras.with_columns(
    DPVOany=(pl.col("APVOany-warm").cast(pl.Boolean) & pl.col("CPVOany-cold").cast(pl.Boolean)).cast(pl.Float32) * 100
)

everything = {}
for jet_name in ["STJ", "EDJ"]:
    everything[jet_name] = prepare_last_step_2(
        props_with_extras,
        spells_list[jet_name],
        summer,
        grams_wr=grams_wr,
        mjo=mjo,
        n_bootstraps=1000,
    )
    # for when, maybe_other in product(["before", "during"], ["", "-other"]):
    #     col1 = pl.col(f"APVOany-warm.{when}{maybe_other}")
    #     col2 = pl.col(f"CPVOany-cold.{when}{maybe_other}")
    #     col3 = f"DPVOany.{when}{maybe_other}"
    #     everything[jet_name] = everything[jet_name].with_columns(**{col3: col1 + col2})

In [ ]:
clusters_da = np.abs(xr.open_dataarray(basepath.joinpath("cluster_df.nc")).load())
clusters_da = clusters_da.interp(lat=np.arange(32, 72, 0.5), method="nearest")

# plt.savefig(f"{FIGURES}/jet_persistence/regions.png")

region_T = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_T_6H.parquet")),
    smooth_clim=31,
    other_index_columns=["region"],
    detrend=True,
)
region_T = region_T.rolling("time", period="3d", group_by="region").agg(
    pl.col("t2m").mean()
)
region_T_ = summer_filter.join(region_T, on="time")
hws = get_spells(
    region_T_,
    pl.col("t2m") > pl.col("t2m").quantile(0.95),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=5),
).sort("region")
region_tp = compute_anomalies_pl(
    pl.read_parquet(basepath.joinpath("region_tp_6H.parquet")),
    smooth_clim=31,
    other_index_columns=["region"],
)
region_tp = region_tp.rolling("time", period="3d", group_by="region").agg(
    pl.col("tp").mean()
)
region_tp_ = summer_filter.join(region_tp, on="time")

pes = get_spells(
    region_tp_,
    pl.col("tp") > pl.col("tp").quantile(0.95),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=3),
).sort("region")
drs = get_spells(
    region_tp_,
    pl.col("tp") < pl.col("tp").quantile(0.05),
    group_by=["region"],
    # fill_holes=datetime.timedelta(hours=18),
    minlen=datetime.timedelta(days=5),
).sort("region")

In [ ]:
props_with_extras.filter(pl.col("is_polar") > 0.6).select(*[pl.corr("vsq", f"EKE300_{n_days}days-core").alias(f"{n_days}days") for n_days in [5, 10, 20, 30]])

In [ ]:
props_with_extras.filter(pl.col("is_polar") > 0.6).select(*[pl.corr("wavinessFV15", f"EKE250_{n_days}days-core").alias(f"{n_days}days") for n_days in [5, 10, 20, 30]])

# props histograms

## forward

In [ ]:
data_vars = [
    "mean_lat",
    "mean_s",
    "wavinessDC16",
    "tp-warm_entrance",
    "APVOany-warm",
    "CPVOany-cold",
    "DPVOany",
    "tp-tropical",
    "t300:grad-core",
    "EKE300_5days-core",
    "hor300_5days-core",
    "t2m-cold",
]

fig = props_vs_quantiles(
    props_with_extras,
    summary_comp,
    data_vars,
    summer,
    100,
    20,
    3,
    4,
    numbering=True,
    clear=False,
    one_ax_each=True,
    col_width=3.2,
)
fig.savefig(f"{FIGURES}/Persistence/candidate_appendix3_forward.pdf")

## Backward

In [ ]:
from jetutils.plots import props_vs_quantiles_bw
data_vars = [
    "mean_lat",
    "mean_s",
    "wavinessDC16",
    "tp-warm_entrance",
    "APVOany-warm",
    "CPVOany-cold",
    "DPVOany",
    "tp-tropical",
    "t300:grad-core",
    "EKE300_5days-core",
    "hor300_5days-core",
    "t2m-cold",
]

fig = props_vs_quantiles_bw(
    props_with_extras,
    summary_comp,
    data_vars,
    summer,
    100,
    20,
    4,
    numbering=True,
    clear=False,
    col_width=3.2,
)
fig.savefig(f"{FIGURES}/Persistence/candidate_appendix3_backward.pdf")

## seasonal cycle of slowness

In [ ]:
from jetutils.plots import plot_seasonal

data_vars = ["mean_theta", "mean_lev", "pers", "slowness"]
fig = plot_seasonal(
    props_catd,
    data_vars,
    nrows=2,
    ncols=2,
    clear=False,
    folder="Variability",
    suffix="_subset",
    numbering=True,
    save=False,
)

# correlation, jet-centred

broken, not so useful

In [ ]:
from jetutils.definitions import to_expr

odir_corr = Path(FIGURES, "Persistence/figure_data_corr")


def corr_agg(a: str | pl.Expr, b: str | pl.Expr):
    a = to_expr(a).replace([float("nan"), float("inf"), -float("inf")], None)
    b = to_expr(b).replace([float("nan"), float("inf"), -float("inf")], None)
    a_ = a - a.mean()
    b_ = b - b.mean()
    cov = (a_ * b_).sum()
    return cov / (a_.pow(2).sum() * b_.pow(2).sum()).sqrt()


n_bootstraps = 100
bs_times = create_bootstrapped_times(
    summer_summary_comp.select(
        "spell", "len", "time", "relative_time", "relative_index"
    ),
    summer,
    n_bootstraps,
).sort("sample_index", "spell", "relative_index")
huh = bs_times.join(
    summer_summary_comp[
        "spell",
        "len",
        "relative_time",
        "relative_index",
        "jet ID",
        "is_polar",
        "catd",
        "jet",
        "slowness_sum",
    ],
    on=["spell", "len", "relative_time", "relative_index"],
)
huh = huh.join(props["time", "jet ID", "is_polar"], on="time")
which_one = (pl.col("is_polar") - pl.col("is_polar_right")).abs().arg_min()
filter_ = huh.group_by("sample_index", "time", "jet ID").agg(
    pl.col("jet ID_right").get(which_one)
)
huh = filter_.join(huh, on=filter_.columns)
huh = huh.drop("is_polar").rename({"is_polar_right": "is_polar"})

for var in ["APVO", "CPVO", "PV330", "PV350", "theta", "tp", "t2m", "hor", "EKE250"]:
    ofile = odir_corr.joinpath(f"{var}.nc")
    if ofile.is_file():
        continue
    df = pl.read_parquet(basepath.joinpath(f"{var}_relative.parquet"))
    df = summer_filter.join(df, on="time")
    out = []
    for s, subhuh in tqdm(
        huh.group_by("sample_index", maintain_order=True), total=n_bootstraps + 1
    ):
        subhuh = (
            subhuh.join(df, on=["time", "jet ID"])
            .group_by("sample_index", "spell", "jet ID", "n", "norm_index")
            .agg(
                pl.col(f"{var}_interp").mean(),
                pl.col("slowness_sum").first(),
                pl.col("jet").first(),
            )
        )
        subhuh = subhuh.group_by("sample_index", "jet", "n", "norm_index").agg(
            corr_agg(f"{var}_interp", "slowness_sum")
        )
        out.append(subhuh)
    df = pl.concat(out)
    to_plot = df.filter(pl.col("sample_index") == n_bootstraps)
    da = polars_to_xarray(to_plot, ["jet", "n", "norm_index"])[f"{var}_interp"]
    pvals = df.group_by("jet", "n", "norm_index").agg(
        (pl.col(f"{var}_interp").rank().last() - 1) / n_bootstraps
    )
    da_pvals = polars_to_xarray(pvals, ["jet", "n", "norm_index"])
    da.to_netcdf(ofile)
    da.to_netcdf(odir_corr.joinpath(f"{var}_pvals.nc"))

# quantile stuff, old, prefer histograms

In [ ]:
from jetutils.definitions import to_expr
from typing import Sequence


def bin_by(
    df: pl.DataFrame,
    by: str | pl.Expr,
    jet: str,
    data_vars: Sequence[str],
    n_q: int = 100,
    lags: pl.Series | None = None,
):
    jet: pl.Expr = pl.col("jet") == jet
    qs = np.linspace(0, 1, n_q + 1).tolist()
    dq = qs[1] - qs[0]
    labels = [f"{int((q + dq / 2) * 100):d}" for q in qs]

    by = to_expr(by)
    filter_ = (
        by.qcut(qs[1:], labels=labels, allow_duplicates=True)
        .cast(pl.String())
        .str.to_integer()
    )
    filter_ = df.filter(jet).drop_nulls(by).drop_nans(by).select("time", catd=filter_)
    if lags is None:
        lags = pl.Series("lags", [datetime.timedelta(0)])
    filter_ = filter_.join(lags.to_frame(), how="cross")
    filter_ = filter_.with_columns(pl.col("time") + pl.col("lags"))
    aggs = {var: pl.col(var).mean() for var in data_vars}
    aggs = {"len": pl.col("time").len()} | aggs
    return (
        df.join(filter_, on="time", how="left")
        .group_by("catd", "jet", "lags")
        .agg(**aggs)
        .sort("catd", "lags", "jet")
    )

## forward

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")
lags = pl.Series("lags", [datetime.timedelta(days=n.item()) for n in np.arange(-4, 5)])
n_q = 31
data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    "wavinessR16",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(
            props_as_df_anoms, by, jet_name, data_vars=data_vars, n_q=n_q, lags=lags
        ).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet_name, lags=datetime.timedelta(days=3))
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(
                ouais["catd"] / 100,
                ouais[data_var],
                label=PRETTIER_VARNAME[data_var],
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet slowsness vs. props. of the {jet_name[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "tp_tropical",
    "tp_gulfstream",
    "t2m_tropical",
    "t2m_east",
    "t2m_west",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(
            props_as_df_anoms, by, jet_name, data_vars=data_vars, n_q=n_q
        ).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet_name)
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(
                ouais["catd"] / 100,
                ouais[data_var],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet slowsness vs. props. of the {jet_name[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [f"t2m_{i}" for i in range(1, 7)]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(
            props_as_df_anoms,
            by,
            jet_name,
            data_vars=data_vars,
            n_q=n_q,
            lags=pl.Series("lags", [datetime.timedelta(days=0)]),
        ).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet_name)
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(
                ouais["catd"] / 100,
                ouais[data_var],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [f"tp_{i}" for i in range(1, 7)]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(
            props_as_df_anoms,
            by,
            jet_name,
            data_vars=data_vars,
            n_q=n_q,
            lags=pl.Series("lags", [datetime.timedelta(days=0)]),
        ).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet_name)
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(
                ouais["catd"] / 100,
                ouais[data_var],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=2, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    "mean_lat",
    "tp-warm",
    "tp-cold",
    "t2m-warm",
    "t2m-cold",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ouais = bin_by(
            props_as_df_anoms,
            by,
            jet_name,
            data_vars=data_vars,
            n_q=n_q,
            lags=pl.Series("lags", [-datetime.timedelta(days=0)]),
        ).drop_nulls("catd")
        ouais = ouais.filter(pl.col("jet") == jet_name)
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ax.plot(
                ouais["catd"] / 100,
                ouais[data_var],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet slowsness vs. props. of the {jet_name[:3]}", pad=4)
fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
data_vars = (
    [
        # "tp-warm",
        # "tp-cold",
        # "t2m-warm",
        # "t2m-cold",
        "mean_lat",
        "mean_theta",
        "mean_s",
        "wavinessR16",
    ]
    + [f"t2m_{i}" for i in range(1, 7)]
    + [f"tp_{i}" for i in range(1, 7)]
)
by = "pers_catd"
n_q = 101
lags = pl.Series("lags", [datetime.timedelta(days=int(n)) for n in np.arange(-10, 11)])

In [ ]:
def the_struct(var):
    return pl.struct(
        at=pl.col("lags").get(pl.col(var).abs().arg_max()),
        val=pl.col(var).get(pl.col(var).abs().arg_max()),
    )


for jet_name in ["STJ", "EDJ"]:
    huh = (
        bin_by(props_as_df_anoms, by, jet_name, data_vars=data_vars, n_q=n_q, lags=lags)
        .drop_nulls("catd")
        .filter(pl.col("jet") == jet_name)
    )
    res = (
        huh.group_by("lags", maintain_order=True)
        .agg(
            **{
                data_var: pds.lin_reg_report("catd", target=data_var)
                .struct.field("beta")
                .first()
                * 100
                for data_var in data_vars
            }
        )
        .select(**{var: the_struct(var) for var in data_vars})
    )
    print(res)

## backwards
quantile of the prop vs absolute value of slowness

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    "wavinessR16",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(
                props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(
                    ["pers", data_var]
                ),
                data_var,
                jet_name,
                data_vars=["pers"],
                n_q=n_q,
            ).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet_name)
            ax.plot(
                ouais["catd"] / 100,
                ouais["pers"],
                label=PRETTIER_VARNAME[data_var],
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 11
data_vars = [
    "tp_tropical",
    "tp_gulfstream",
    "t2m_tropical",
    "t2m_east",
    "t2m_west",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(
                props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(
                    ["pers", data_var]
                ),
                data_var,
                jet_name,
                data_vars=["pers"],
                n_q=n_q,
            ).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet_name)
            ax.plot(
                ouais["catd"] / 100,
                ouais["pers"],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [f"t2m_{i}" for i in range(1, 7)]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(
                props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(
                    ["pers", data_var]
                ),
                data_var,
                jet_name,
                data_vars=["pers"],
                n_q=n_q,
            ).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet_name)
            ax.plot(
                ouais["catd"] / 100,
                ouais["pers"],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [f"tp_{i}" for i in range(1, 7)]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(
                props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(
                    ["pers", data_var]
                ),
                data_var,
                jet_name,
                data_vars=["pers"],
                n_q=n_q,
            ).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet_name)
            ax.plot(
                ouais["catd"] / 100,
                ouais["pers"],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

In [ ]:
from itertools import pairwise

props_as_df_anoms = compute_anomalies_pl(
    props_with_extras, other_index_columns=["jet"], standardize=True
)
props_as_df_anoms = summer_filter.join(props_as_df_anoms, on="time")

n_q = 31
data_vars = [
    "mean_lat",
    "tp-warm_entrance",
    "tp-cold",
    "t2m-warm",
    "t2m-cold",
]
bys = ["pers_catd"]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
qs = np.linspace(0, 1, n_q)
fig, axes = plt.subplots(
    len(bys),
    2,
    figsize=(10, 5),
    sharex="all",
    sharey="row",
    gridspec_kw={"wspace": 0.1, "left": 0.1},
)
axes = np.atleast_2d(axes)
for i, by in enumerate(bys):
    for j, jet_name in enumerate(["STJ", "EDJ"]):
        ax = axes[i, j]
        ax.axhline(0.0, color="black")
        for k, data_var in enumerate(data_vars):
            ouais = bin_by(
                props_as_df_anoms.drop_nans(["pers", data_var]).drop_nulls(
                    ["pers", data_var]
                ),
                data_var,
                jet_name,
                data_vars=["pers"],
                n_q=n_q,
            ).drop_nulls("catd")
            ouais = ouais.filter(pl.col("jet") == jet_name)
            ax.plot(
                ouais["catd"] / 100,
                ouais["pers"],
                label=PRETTIER_VARNAME.get(data_var, data_var),
                color=colors_props[k],
                lw=2,
            )
        ax.set_title(f"Jet persistence vs. props. of the {jet_name[:3]}", pad=4)
# fig.supylabel("Standardized anomaly of the jet property", wrap=True)
axes[-1, 0].legend(ncol=1, fontsize=15)
axes[-1, -1].set_xlim([0, 1])

# jet pos

In [ ]:
jet_pos_da = jet_position_as_da(jets)

plt.ion()
MYPURPLES = LinearSegmentedColormap.from_list(
    "mypurples", ["#ffffff", COLORS_EXT[4], COLORS_EXT[5]]
)
MYPINKS = LinearSegmentedColormap.from_list(
    "mypinks", ["#ffffff", COLORS_EXT[7], COLORS_EXT[8]]
)

to_plot_1 = []
to_plot_2 = []
for month in range(1, 13):
    this_da = jet_pos_da.sel(time=jet_pos_da.time.dt.month == month)
    to_plot_1.append((this_da < 0.5).mean("time"))
    to_plot_2.append((this_da > 0.5).mean("time"))

clu = Clusterplot(3, 4, get_region(jet_pos_da), row_height=3.4)
im, kwargs = clu.add_contourf(
    [tp2 * 100 for tp2 in to_plot_2],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.gothic_r,
    draw_cbar=False,
    titles=MONTH_NAMES,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Eddy-driven jet occurence [$\%$]", fontsize=13)
im, kwargs = clu.add_contourf(
    [tp1 * 100 for tp1 in to_plot_1],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.flamingo_r,
    draw_cbar=False,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Subtropical jet occurence [\%]", fontsize=13)
clu.resize_relative([1.0, 1.06])

In [ ]:
jet_pos_da_eps = jet_position_as_da(
    spells_list["EDJ"]
    .join(jets.cast({"time": pl.Datetime("ms")}), on="time")
    .drop("spell", "relative_index")
)
plt.ion()
MYPURPLES = LinearSegmentedColormap.from_list(
    "mypurples", ["#ffffff", COLORS_EXT[4], COLORS_EXT[5]]
)
MYPINKS = LinearSegmentedColormap.from_list(
    "mypinks", ["#ffffff", COLORS_EXT[7], COLORS_EXT[8]]
)

to_plot_1 = []
to_plot_2 = []
for month in range(7, 10):
    this_da = jet_pos_da_eps.sel(time=jet_pos_da_eps.time.dt.month == month)
    to_plot_1.append((this_da < 0.5).mean("time"))
    to_plot_2.append((this_da > 0.5).mean("time"))

clu = Clusterplot(1, 3, get_region(jet_pos_da_eps), row_height=6)
im, kwargs = clu.add_contourf(
    [tp2 * 100 for tp2 in to_plot_2],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.gothic_r,
    draw_cbar=False,
    titles=MONTH_NAMES[6:],
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Eddy-driven jet occurence [$\%$]", fontsize=13)
im, kwargs = clu.add_contourf(
    [tp1 * 100 for tp1 in to_plot_1],
    transparify=1,
    levels=7,
    q=0.995,
    cmap=colormaps.flamingo_r,
    draw_cbar=False,
)
cbar = clu.fig.colorbar(im, ax=clu.axes, shrink=0.99, pad=0.01)
cbar.ax.set_ylabel(r"Subtropical jet occurence [\%]", fontsize=13)
clu.resize_relative([1.0, 1.06])

# indiv spells

In [ ]:
from matplotlib.cm import ScalarMappable
winner_names = pl.DataFrame({"winner": list(range(5)), "name": ["No", "GB", "AL", "AR", "SB"]})

def question_func(question: tuple[str]) -> pl.Expr:
    col, which = question
    col = pl.col(col)
    if which == "yesno":
        return col > 0
    if which == "blocked":
        return col.is_in([2, 4]).fill_null(False)
    if which == "match":
        return col.is_in([2, 3, 4]).fill_null(False)
    test_against = col.filter(pl.col("spell") == -2)
    esteedee = 0.5 * col.filter(pl.col("spell") == -3)
    if which == "above":
        return (col > test_against + esteedee).fill_null(False)
    elif which == "below":
        return (col < test_against - esteedee).fill_null(False)
    return col


questions_str_stj = [
    ("regime", "blocked"),
    ("theta:grad-core", "above"),
    ("EKE250_5days-core", "below"),
    ("mean_lat.other", "above"),
    ("mean_s.other", "below"),
    ("t2m-cold", "above"),
    # ("tp_anom-tropical_indian", "above"),
    ("tp_anom-tropical_pacific", "above"),
    ("tp_anom-tropical_atlantic", "above"),
]

questions_str_edj = [
    ("regime", "blocked"),
    ("t300:grad-core", "above"),
    ("EKE300_5days-core", "below"),
    # ("hor300_5days-core", "above"),
    # ("vert300_5days-core", "above"),
    # ("lat_over_europe", "above"),
    ("mean_s", "above"),
    ("mean_lat", "below"),
    ("DPVOany", "above"),
    # ("APVOany-warm", "above"),
    ("CPVOany-cold", "above"),
    ("tp-warm_entrance", "above"),
]

plot_kwargs = {
    "STJ": (
        questions_str_stj,
        (
            question_func(("tp_anom-tropical_pacific.during", "above"))
            | question_func(("tp_anom-tropical_atlantic.during", "above")),
            # | question_func(("tp_anom-tropical_indian.during", "above")),
            question_func(("t2m-cold.during", "above")),
        ),
        pl.col("regime.during").is_in([2, 4]).fill_null(False),
    ),
    "EDJ": (
        questions_str_edj,
        (
            question_func(("tp-warm_entrance.during", "above")),
            question_func(("CPVOany-cold.during", "above")),
        ),
        pl.col("CPVOany-cold.before"),
    ),
}
groupnames_stj = [
    r"\textbf{Low precip, low temp.}",
    r"\textbf{Low precip, high temp.}",
    r"\textbf{High precip, low temp.}",
    r"\textbf{High precip, high temp.}",
]
groupnames_edj = [
    r"\textbf{Low precip, no CWB}",
    r"\textbf{Low precip, CWB}",
    r"\textbf{High precip, no CWB}",
    r"\textbf{High precip, CWB}",
]
groupnames = {"STJ": groupnames_stj, "EDJ": groupnames_edj}
cmap = LinearSegmentedColormap.from_list("", ["white", COLORS[3]])
norm = BoundaryNorm([0.0, 0.5, 1], cmap.N, extend="neither")
im = ScalarMappable(norm, cmap=cmap)
colors_groups = colormaps.pastel([2, 3, 4, 6, 7, 8, 0, 5, 1])

for jet_name, values in plot_kwargs.items():
    questions_str, grouping, sorting = values
    df: pl.DataFrame = everything[jet_name]
    groups = df.select(
        "spell",
        group=pl.concat_str(*[g.cast(pl.UInt8()) for g in grouping]).str.to_integer(
            base=2
        ),
    )
    df = df.join(groups, on="spell", how="left").sort(
        pl.col("spell") == -2, pl.col("spell") == -1, pl.col("group"), sorting
    )
    groups = df.select("spell", "group")
    

    ## xticks
    spell_labels = []
    spell_labels_colors = []
    for i_spell, group in groups.filter(pl.col("spell") >= 0).iter_rows():
        spell_dates = spells_list[jet_name].filter(pl.col("spell") == i_spell)["time"]
        year = str(spell_dates.dt.year().mode().item()).zfill(4)
        first_date, last_date = (
            spell_dates.dt.strftime("%d/%m").gather([0, -1]).to_list()
        )
        spell_labels.append(r"$\mathbf{" + f"{year}, {first_date}-{last_date}" + r"}$")
        spell_labels_colors.append(colors_groups[group])

    spell_labels.append("Episode mean")
    # spell_labels.append("Mean clim")
    spell_labels_colors.extend(["black", "black"])

    max_len_xticklabel = 0
    dummy_fig, dummy_ax = plt.subplots()
    r = dummy_fig.canvas.get_renderer()
    for label in spell_labels:
        fontsize = plt.rcParams["xtick.labelsize"]
        t = dummy_ax.text(0, 0, label, fontsize=fontsize)
        bb = t.get_window_extent(renderer=r)
        width = bb.width / dummy_fig.get_dpi()
        height = bb.height / dummy_fig.get_dpi()
        max_len_xticklabel = max(max_len_xticklabel, width)
    plt.close(dummy_fig)

    yticklabels = []
    for key, direction in questions_str:
        if "." in key and key.split(".")[-1] == "other":
            rest, other = key.split(".")
        else:
            rest = key
            other = None
        elements = rest.split("-")
        varname = elements[0]
        label = PRETTIER_VARNAME.get(varname.rstrip("_anom"), varname)
        if ":" in varname and varname.split(":")[-1] == "grad":
            varname_ = varname.split(":")[0]
            label = f"Grad of {PRETTIER_VARNAME.get(varname_, varname_)}"
        if other is not None:
            otherjet = "EDJ" if jet_name == "STJ" else "STJ"
            label = f"{label} of the {otherjet}"
        if len(elements) == 2:
            where_ = elements[1]
            if where_[:4] == "trop":
                where_ = where_.split("_")
                where_ = where_[0] + " " + where_[1][0].upper() + where_[1][1:]
            else:
                where_ = ", ".join(where_.split("_"))
            label = f"{label}, {where_}"
        label = f"{label}, {direction} ?"
        yticklabels.append(label)

    max_len_yticklabel = 0
    dummy_fig, dummy_ax = plt.subplots()
    r = dummy_fig.canvas.get_renderer()
    for label in yticklabels:
        fontsize = plt.rcParams["xtick.labelsize"]
        t = dummy_ax.text(0.5, 0.5, label, fontsize=fontsize)
        bb = t.get_window_extent(renderer=r)
        width = bb.width / dummy_fig.get_dpi()
        height = bb.height / dummy_fig.get_dpi()
        max_len_yticklabel = max(max_len_yticklabel, width)
    plt.close(dummy_fig)

    ## main
    x = np.arange(groups.shape[0] - 2)
    y = np.arange(len(questions_str))
    box_side = 0.3
    wspace = 0.02
    hspace = 0.04
    other_ratio = 0.3
    fig_width = (max_len_yticklabel + (box_side * len(x)) * (1 + other_ratio)) * (
        1 + wspace
    )
    fig_height = max_len_xticklabel + (box_side * len(y) * 2) * (1 + 0.01) + wspace
    bottom = max_len_xticklabel / fig_height
    left = max_len_yticklabel / fig_width
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(fig_width, fig_height),
        width_ratios=(1, other_ratio),
        gridspec_kw=dict(bottom=bottom, left=left, wspace=wspace, hspace=hspace),
    )
    axes = axes
    for axs, when in zip(axes, ["during", "before"]):
        questions = {}
        pvals_keys = []
        for key, huh in questions_str:
            if "." in key and key.split(".")[-1] == "other":
                main, other = key.split(".")
                newkey = f"{main}.{when}-{other}"
            else:
                newkey = f"{key}.{when}"
            questions[newkey] = question_func((newkey, huh))
            pvals_keys.append(f"{newkey}.pvals")
        to_plot = (
            df.select("spell", **questions)
            .filter(pl.col("spell") >= -1)
            .drop("spell")
            .to_numpy()
            .T
        )
        to_plot_sum = (
            df.select("spell", **questions)
            .filter(pl.col("spell") >= 0)
            .drop("spell")
            .to_numpy()
            .sum(axis=0)
        )
        where_vertical_lines = (
            groups.filter(pl.col("spell") >= -1)
            .with_row_index()
            .filter(pl.col("group").diff().fill_null(0).cast(pl.Boolean()))["index"]
            - 0.5
        )
        ax = axs[0]
        # ax.set_aspect("equal")
        ax.pcolormesh(x, y, to_plot, norm=norm, cmap=cmap, edgecolor="black", lw=0.5)
        for where_vertical_line in where_vertical_lines:
            ax.axvline(where_vertical_line, ls="solid", lw=5, color="black")
        # fig.colorbar(im, ax=ax, shrink=0.36, pad=0.01)


        if when == "before":
            ax.set_xticks(x, labels=spell_labels, rotation=90)
        else:
            ax.set_xticks([])
        for ticklabel in ax.yaxis.get_ticklabels():
            if when == "during":
                ticklabel.set_color("black")
            else:
                ticklabel.set_color("lightslategrey")
        # ax.set_xlabel(f"Persistent episode of the {jet}")

        ax.set_yticks(y, labels=yticklabels)
        for color, ticklabel in zip(spell_labels_colors, ax.xaxis.get_ticklabels()):
            ticklabel.set_color(color)
            ticklabel.set_color(color)
            
        ax = axs[1]
        ax.barh(y, to_plot_sum, color=COLORS[3], height=1.0, edgecolor="black", lw=1.0)
        ax.set_xlim([0, 30])
        xticks = np.arange(0, 35, 5)
        if when == "before":
            ax.set_xticks(xticks)
            ax.set_xlabel("Number of lifecycles")
        else:
            ax.set_xticks(xticks, labels=["" for _ in xticks])
            ax.tick_params(axis="x", length=0)
        ax.set_yticks([])
        ax.set_ylim(axes[0, 0].get_ylim())
        ax.grid(True, axis="x")

    x_anchor = 0.
    y_anchor = 0.03
    height_per_height = 0.052
    toptext = y_anchor + (len(groupnames[jet_name]) + 0.4) * height_per_height
    width = 0.185
    height = (len(groupnames[jet_name]) - 0.3) * height_per_height + 0.03
    phax = fig.add_axes([x_anchor, y_anchor, width, height])
    phax.xaxis.set_visible(False)
    phax.yaxis.set_visible(False)
    phax.set_zorder(1000)
    phax.patch.set_alpha(0.0)
    phax.patch.set_color("w")
    # fig.text(
    #     x_anchor + 0.01,
    #     toptext,
    #     r"\textbf{Groups}:",
    #     color="black",
    #     fontsize=20,
    # )
    txtkwargs = {"fontsize": 16}
    for i, groupname in enumerate(groupnames[jet_name]):
        fig.text(
            x_anchor + 0.01,
            toptext - (i + 1) * height_per_height,
            groupname,
            color=colors_groups[i],
            **txtkwargs,
        )
    # t1 = r"Above: $\psi > \overline{\psi} + 0.5 \times \mathrm{std}(\psi)$"
    # t1 = fig.text(0.74, 0.19, t1, ha="left", va="center")
    # t2 = r"Below: $\psi < \overline{\psi} - 0.5 \times \mathrm{std}(\psi)$"
    # t2 = fig.text(0.74, 0.14, t2, ha="left", va="center")
    # phax = fig.add_axes([0.74 - 0.003, 0.11, 0.17, 0.115])
    # phax.xaxis.set_visible(False)
    # phax.yaxis.set_visible(False)
    # phax.set_zorder(1000)
    # phax.patch.set_alpha(0.0)
    # phax.patch.set_color("w")
    fig.savefig(f"{FIGURES}/Persistence/last_v7_{jet_name}.pdf")

# interp

In [ ]:
variables = [
    "theta",
    "theta:grad",
    "t2m",
    "tp",
    "eady", 
    "eady_s",
]
for level in [200, 225, 250, 300, 350, 850]:
    variables.append(f"t{level}")
    variables.append(f"t{level}:grad")
for n_days, level, var in product(
    [5, 10, 20, 30], [250, 300], ["EKE", "hor", "vert", "up", "vp"]
):
    variables.append(f"{var}{level}_{n_days}days")
for var in ["APVO", "CPVO"]:
    for level in [320, 330, 340, 350, "any"]:
        variables.append(f"{var}{level}")
# for level in [320, 330, 340, 350]:
#     for var, days in product(["EKE", "EMFconv"], [10, 20, 30]):
#         variables.append(f"{var}{level}_{days}days")
#     variables.append(f"PV{level}")
#     variables.append(f"PV{level}:grad")

In [ ]:
create_all_relative_plots(
    spells_list,
    variables,
    basepath,
    odir=Path(FIGURES, "Persistence/figure_data"),
    season=summer,
    n_bootstraps=100,
)

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/RWB")
odir.mkdir(exist_ok=True)

variables = {}
for level in [320, 330, 340, 350, "any"]:
    variables[f"APVO{level}:clim"] = [12, colormaps.cet_l_bmy_r, [0, 50]]
    variables[f"APVO{level}:anom"] = [16, colormaps.BlWhRe, [-20, 20]]
    variables[f"CPVO{level}:clim"] = [12, colormaps.cet_l_bmy_r, [0, 40]]
    variables[f"CPVO{level}:anom"] = [16, colormaps.BlWhRe, [-10, 10]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=4,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
    # im = PIL.Image.open(odir.joinpath(f"{jet_name}.png"))
    # im.save(odir.joinpath(f"{jet_name}convd.pdf"), "PDF", quality=100)
    # plt.close()

In [ ]:
level = 300
n_days = 5
df = pl.read_parquet(basepath.joinpath(f"v{level}_relative.parquet"))
df = df.with_columns((pl.col(f"v{level}_interp") ** 2 * 0.5).alias(f"vsq{level}_interp"))
df.write_parquet(basepath.joinpath(f"vsq{level}_relative.parquet"))

df = pl.read_parquet(basepath.joinpath(f"vp{level}_{n_days}days_relative.parquet"))
df = df.with_columns((pl.col(f"vp{level}_{n_days}days_interp") ** 2 * 0.5).alias(f"vpsq{level}_{n_days}days_interp"))
df.write_parquet(basepath.joinpath(f"vpsq{level}_{n_days}days_relative.parquet"))

df = pl.read_parquet(basepath.joinpath(f"vjp{level}_{n_days}days_relative.parquet"))
df = df.with_columns((pl.col(f"vjp{level}_{n_days}days_interp") ** 2 * 0.5).alias(f"vjpsq{level}_{n_days}days_interp"))
df.write_parquet(basepath.joinpath(f"vjpsq{level}_{n_days}days_relative.parquet"))

create_all_relative_plots(
    spells_list,
    [f"vsq{level}", f"vpsq{level}_{n_days}days", f"vjpsq{level}_{n_days}days"],
    basepath,
    odir=Path(FIGURES, "Persistence/figure_data"),
    season=summer,
    n_bootstraps=100,
)

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/is_eke_v")
odir.mkdir(exist_ok=True)
n_days = 5
variables = {}
level = 300
variables[f"EKE{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
variables[f"EKE{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
variables[f"vsq{level}:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
variables[f"vsq{level}:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
variables[f"vpsq{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
variables[f"vpsq{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
variables[f"vjpsq{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
variables[f"vjpsq{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=4,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
# plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddyv3")
odir.mkdir(exist_ok=True)
n_days = 5
variables = {}
for level in [850, 300, 250]:
    variables[f"EKE{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
    variables[f"EKE{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
    variables[f"hor{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"hor{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=6,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
# plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddyv4")
odir.mkdir(exist_ok=True)
level = 250
variables = {}
for n_days in [5, 10, 20, 30]:
    variables[f"EKE{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
    variables[f"EKE{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
    variables[f"hor{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"hor{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=6,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
# plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddyv4")
odir.mkdir(exist_ok=True)
level = 300
variables = {}
for n_days in [5, 10, 20, 30]:
    variables[f"EKE{level}_{n_days}days:clim"] = [12, colormaps.cet_l_bmy_r, [0, 150]]
    variables[f"EKE{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-50, 50]]
    variables[f"hor{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"hor{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:clim"] = [12, colormaps.BlWhRe, [-300, 300]]
    variables[f"vert{level}_{n_days}days:anom"] = [16, colormaps.BlWhRe, [-300, 300]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=6,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
# plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/uppert")
odir.mkdir(exist_ok=True)
variables = {}
for level in [850, 350, 300, 250, 225, 200]:
    variables[f"t{level}:clim"] = [12, colormaps.bilbao_r, [200, 300]]
    variables[f"t{level}:anom"] = [16, colormaps.BlWhRe, [-5, 5]]
    variables[f"t{level}:clim_grad"] = [12, colormaps.bilbao_r, [0, 20]]
    variables[f"t{level}:anom_grad"] = [16, colormaps.BlWhRe, [-5, 5]]
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=4,
        square_len=3.3,
        transpose=True,
        pad=0,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
# plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eady_growth")
odir.mkdir(exist_ok=True)
variables = {
    "eady:clim": [12, colormaps.bilbao_r, [0, 2]],
    "eady:anom": [12, colormaps.BlWhRe, [-.3, .3]],
    "eady_s:clim": [12, colormaps.bilbao_r, [0, 2]],
    "eady_s:anom": [12, colormaps.BlWhRe, [-.3, .3]],
}

for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
        alpha=0.05,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/ground")
odir.mkdir(exist_ok=True)
variables = {
    "t2m:clim": [6, colormaps.bilbao_r, [280, 300]],
    "t2m:anom": [8, colormaps.BlWhRe, [-4, 4]],
    "tp:clim": [9, colormaps.freeze_r, [0, 40]],
    "tp:anom": [9, colormaps.brbg, [-12, 12]],
}

for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
        alpha=0.05,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddies")
odir.mkdir(exist_ok=True)

variables = {
    "EKE250_5days:clim": [12, colormaps.cet_l_bmy_r, [0, 300]],
    "EKE250_5days:anom": [12, colormaps.BlWhRe, [-60, 60]],
    "hor250_5days:clim": [12, colormaps.BlWhRe, [-400, 400]],
    "hor250_5days:anom": [12, colormaps.BlWhRe, [-400, 400]],
}
quiv_dict = dict(width=.005, headwidth=3, pivot="mid", scale=40)
quivers = {
    "EKE250_5days:clim": ["up250_5days:clim", "vp250_5days:clim", quiv_dict],
    "EKE250_5days:anom": ["up250_5days:anom", "vp250_5days:anom", quiv_dict | {"scale": 10}],
}
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hide",
        n_col=2,
        square_len=3.3,
        transpose=True,
        quivers=quivers,
    )
    fig.savefig(odir.joinpath(f"{jet_name}.pdf"))
    # plt.close()

In [ ]:
ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/eddies")
odir.mkdir(exist_ok=True)

variables = {
    "up250_10days:clim": [12, colormaps.cet_l_bmy_r, [0, 10]],
    "up250_10days:anom": [12, colormaps.BlWhRe, [-3, 3]],
    "vp250_10days:clim": [12, colormaps.BlWhRe, [-3, 3]],
    "vp250_10days:anom": [12, colormaps.BlWhRe, [-3, 3]],
}
quiv_dict = dict(width=.005, headwidth=3, pivot="mid", scale=40)
for jet_name in ["STJ", "EDJ"]:
    fig = plot_interp(
        variables,
        "",
        ipath,
        jet_name,
        handle_pvals="hatch",
        n_col=2,
        square_len=3.3,
        transpose=True,
    )
    fig.savefig(odir.joinpath(f"{jet_name}_uvp.pdf"))
    # plt.close()

In [ ]:
df = pl.read_parquet(basepath.joinpath("up250_5days_relative.parquet"))
df2 = pl.read_parquet(basepath.joinpath("vp250_5days_relative.parquet"))

In [ ]:
df.join(jets[["time", "jet ID", "u", "v"]], on="tim")

In [ ]:
# STJ upper-levels

def reduce_cmap_(cmap, vmin, vmax):
    newN = int((vmax - vmin) * cmap.N) 
    return LinearSegmentedColormap.from_list("", cmap(np.linspace(vmin, vmax, newN)))

ipath = Path(f"{FIGURES}/Persistence/figure_data")
odir = Path(f"{FIGURES}/Persistence/upper_level")
odir.mkdir(exist_ok=True)

cmap_div = reduce_cmap_(colormaps.BlWhRe, 0.15, 0.85)
variables = {
    # "t250:clim": [6, colormaps.bilbao_r, [220, 240]],
    # "t250:anom": [7, cmap_div, [-2, 2]],
    # "t250:clim_grad": [6, colormaps.bilbao_r, [0, 12]],
    # "t250:anom_grad": [7, cmap_div, [-4, 4]],
    "theta:clim": [8, colormaps.bilbao_r, [330, 370]],
    "theta:anom": [8, cmap_div, [-8, 8]],
    "theta:clim_grad": [7, colormaps.bilbao_r, [0, 60]],
    "theta:anom_grad": [7, cmap_div, [-20, 20]],
    # "APVOany:clim": [7, colormaps.cet_l_bmy_r, [0, 60]],
    # "APVOany:anom": [8, colormaps.BlWhRe, [-20, 20]],
    "EKE250_5days:clim": [6, colormaps.cet_l_bmy_r, [0, 100]],
    "EKE250_5days:anom": [7, cmap_div, [-40, 40]],
}
jet_name = "STJ"
fig = plot_interp(
    variables,
    "",
    ipath,
    jet_name,
    handle_pvals="hatch",
    n_col=2,
    square_len=3.3,
    transpose=True,
    alpha=0.05,
)
fig.savefig(odir.joinpath(f"{jet_name}_v2.pdf"))

variables = {
    "t300:clim": [6, colormaps.bilbao_r, [220, 240]],
    "t300:anom": [7, cmap_div, [-2, 2]],
    "tp:clim": [6, colormaps.freeze_r, [0, 50]],
    "tp:anom": [7, colormaps.brbg, [-12, 12]],
    "t300:clim_grad": [6, colormaps.bilbao_r, [0, 12]],
    "t300:anom_grad": [7, cmap_div, [-3, 3]],
    # "theta:clim_grad": [8, colormaps.bilbao_r, [0, 40]],
    # "theta:anom_grad": [10, cmap_div, [-10, 10]],
    "APVOany:clim": [6, colormaps.cet_l_bmy_r, [0, 60]],
    "APVOany:anom": [7, cmap_div, [-10, 10]],
    "EKE300_5days:clim": [6, colormaps.cet_l_bmy_r, [0, 200]],
    "EKE300_5days:anom": [7, cmap_div, [-30, 30]],
    "CPVOany:clim": [6, colormaps.cet_l_bmy_r, [0, 30]],
    "CPVOany:anom": [7, cmap_div, [-10, 10]],
}
jet_name = "EDJ"
fig = plot_interp(
    variables,
    "",
    ipath,
    jet_name,
    handle_pvals="hatch",
    n_col=4,
    square_len=3.3,
    transpose=True,
    alpha=0.1,
)
fig.savefig(odir.joinpath(f"{jet_name}_v2.pdf"))
# plt.close()

# when spells

## full grid

In [ ]:
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) - 0.3, f"{year}", fontsize=10)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells.pdf")

## stats

In [ ]:
figure = plt.figure(figsize=(13, 5), constrained_layout=True)
subfigs = figure.subfigures(1, 4)


def annotate(ax, annotation, coords=None):
    if coords is None:
        coords = (0.97, 0.97)
    ax.annotate(
        annotation,
        coords,
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
    )
    return ax


# distances, slowness_sum, lifetime lengths, episode distrib
df = summer_summary_comp
cross_summer = summer_filter.join(cross, on="time")
jet_name = (
    pl.when(pl.col("is_polar") < 0.5).then(pl.lit("STJ")).otherwise(pl.lit("EDJ"))
)
cross_summer = cross_summer.filter(pl.col("dis_polar") < dis_polar_thresh).with_columns(
    jet=jet_name
)
jets = ["STJ", "EDJ"]
colors = [COLORS[2], COLORS[1]]

## cross distances, no smoothing
fig = subfigs[0]
axes = fig.subplots(2, 1, sharex="col", sharey="col")
letters = "ab"
for jet_name, color, ax, letter in zip(jets, colors, axes, letters):
    df_ = cross_summer.filter(pl.col("jet") == jet_name)
    ax.hist(
        df_["dist"] / 1e6,
        color=color,
        bins=np.linspace(0, 10, 31),
        density=True,
        edgecolor="white",
        linewidth=1,
    )
    ax.axvline(dist_threshold / 1e6, color="black", ls="dashed")
    ax = annotate(ax, letter)
fig.supylabel("Distance density")
axes[1].set_xlabel("Dist. between jets [1000 km]")
axes[1].set_xticks(np.arange(11))

## Persistence
fig = subfigs[1]
axes = fig.subplots(2, 1, sharex="col", sharey="col")
letters = "cd"
for jet_name, color, ax, letter in zip(jets, colors, axes, letters):
    df_ = (
        summer_summary_comp.group_by("spell")
        .agg(pl.col("slowness_sum").first(), pl.col("jet").first())
        .filter(pl.col("jet") == jet_name)
    )
    ax.hist(
        df_["slowness_sum"],
        color=color,
        bins=np.linspace(0, 2, 31),
        density=True,
        log=True,
        edgecolor="white",
        linewidth=1,
    )

    cutoff = spells_list[jet_name]["slowness_sum"].min()
    ax.axvline(cutoff, color="black", ls="dashed")
    ax = annotate(ax, letter)
fig.supylabel("Persistence density")
axes[1].set_xlabel("Persistence [s/m]")
axes[1].set_xticks(np.linspace(0, 2, 5))

## Lengths
fig = subfigs[2]
axes = fig.subplots(2, 1, sharex="col", sharey="col")
letters = "ef"
for jet_name, color, ax, letter in zip(jets, colors, axes, letters):
    df_ = (
        summer_summary_comp.group_by("spell")
        .agg(pl.col("len").first(), pl.col("jet").first())
        .filter(pl.col("jet") == jet_name)
    )
    ax.hist(
        df_["len"] / 4,
        color=color,
        bins=np.linspace(0, 10, 31),
        density=False,
        log=True,
        edgecolor="white",
        linewidth=1,
    )

    cutoff = spells_list[jet_name]["len"].min() / 4
    ax.axvline(cutoff, color="black", ls="dashed")
    ax = annotate(ax, letter)
fig.supylabel("Lifetime bin count")
axes[1].set_xlabel("Lifetime [day]")
axes[1].set_xticks(np.arange(11))

## When PEs
fig = subfigs[3]
axes = fig.subplots(2, 1, sharex="col", sharey="col")
letters = "gh"
huh = (
    summer.dt.ordinal_day()
    .unique()
    .sort()
    .to_frame()
    .with_columns(
        time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time")
    )
)
bins = pl.datetime_range(
    datetime.datetime(1959, 6, 1),
    datetime.datetime(1959, 10, 1),
    "7d",
    closed="left",
    eager=True,
)
for jet_name, color, ax, letter in zip(jets, colors, axes, letters):
    df_ = spells_list[jet_name].select(
        time=pl.datetime(year=1959, month=1, day=1, time_unit="ms")
        + (
            pl.col("time")
            - pl.datetime(year=pl.col("time").dt.year(), month=1, day=1, time_unit="ms")
        )
    )
    ax.hist(df_["time"], bins=bins, color=color, edgecolor="white", linewidth=1)
    ax = annotate(ax, letter)

ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
ax.xaxis.set_tick_params(labelsize=11, rotation=30)
ticks = ax.get_xticks()
ticklabels = ax.get_xticklabels()
ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
fig.supylabel("Episode bin count")

figure.savefig(f"{FIGURES}/Persistence/stats.pdf")

# Surface ts + extremes

## grids

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = hws.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+hw.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = pes.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+pe.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
regions_I_want = [2, 3]
plt.style.use(["seaborn-v0_8-darkgrid", STYLE_SHEET])
colors = [COLORS[2], COLORS[1]]

fig, ax = plt.subplots(figsize=(1, 0.5), dpi=100)
ax.axis("off")
for spell_name, color in zip(spells_list, colors):
    ax.plot([], [], color=color, lw=3, label=spell_name)
for region_name, color, ir in zip(DUNCANS_REGIONS_NAMES, colors_regions, range(1, 7)):
    if ir not in regions_I_want:
        continue
    ax.plot([], [], color=color, lw=3, label=region_name)
ax.legend()
plt.show()
min_summer, max_summer = (
    summer.dt.ordinal_day().unique().first(),
    summer.dt.ordinal_day().unique().last(),
)
fig, axes = plt.subplots(
    8,
    8,
    figsize=(9, 4),
    gridspec_kw=dict(wspace=0.05, hspace=0.05, left=0, right=1, bottom=0, top=1),
    subplot_kw=dict(
        xticks=[],
        yticks=[],
        xlim=[-1, max_summer - min_summer],
        ylim=[-1, len(colors) + len(regions_I_want) + 1],
    ),
)
axes = axes.ravel()
for ax, year in zip(axes, YEARS):
    ax.text(70, len(spells_list) + len(regions_I_want) - 0.3, f"{year}", fontsize=10)
    for j, ir in enumerate(regions_I_want):
        region_name = DUNCANS_REGIONS_NAMES[ir]
        color = colors_regions[ir - 1]
        j = j + len(spells_list)
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=color,
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = drs.filter(pl.col("time").dt.year() == year, pl.col("region") == ir)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=color, lw=3)
    for j, (name_, spell) in enumerate(spells_list.items()):
        ax.plot(
            [-1, max_summer - min_summer],
            [j, j],
            color=colors[j],
            lw=0.5,
            ls="solid",
            alpha=0.5,
        )
        spell_ = spell.filter(pl.col("time").dt.year() == year)
        if len(spell_) == 0:
            continue
        for s, indiv_spell in spell_.group_by("spell"):
            x = [
                indiv_spell["time"].dt.ordinal_day().first() - min_summer,
                indiv_spell["time"].dt.ordinal_day().last() - min_summer,
            ]
            y = [j, j]
            ax.plot(x, y, color=colors[j], lw=3)
plt.show()
plt.style.use(["default", STYLE_SHEET])
fig.savefig(f"{FIGURES}/Persistence/when_spells+dr.pdf")

## relative time

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
figure = plt.figure(figsize=(10, 4.5), constrained_layout=True)
subfigs = figure.subfigures(1, 2, wspace=0.00, width_ratios=[6.5, 3.5])
n_bootstraps = 1000
with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
    fig = subfigs[0]
    axes = fig.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = spells_list[spell_of].drop("jet ID")
        spells = extend_spells(spells, time_before=datetime.timedelta(days=4))
        times = create_bootstrapped_times(spells, summer, n_bootstraps)
        for j, (name, df) in enumerate({"t2m": region_T, "tp": region_tp}.items()):
            ax = axes[i, j]
            region_ts_masked = times.join(df, on="time").sort(
                "sample_index", "region", "spell", "relative_index"
            )
            x, alive_spells = (
                region_ts_masked.filter(
                    pl.col("region") == 1, pl.col("sample_index") == n_bootstraps
                )
                .group_by("relative_index")
                .agg(pl.col("spell").n_unique())
                .sort("relative_index")
                .to_numpy()
                .T
            )
            region_ts_masked = region_ts_masked.group_by(
                "sample_index", "region", "relative_index", maintain_order=True
            ).agg(pl.col(name).mean())
            pvals = region_ts_masked.group_by(
                "region", "relative_index", maintain_order=True
            ).agg(pl.col(name).rank().last() / n_bootstraps)
            pvals = pvals.with_columns(
                pl.min_horizontal(pl.col(name), 1 - pl.col(name)) * 2
            )
            region_ts_masked = region_ts_masked.group_by(
                "region", "relative_index", maintain_order=True
            ).agg(pl.col(name).last())

            x = x / 4
            filter_ = alive_spells > 15
            ax.axhline(0, color="black")
            ax.axvline(0, color="black")
            for reg, huh_ in region_ts_masked.group_by("region", maintain_order=True):
                y = huh_[name].to_numpy()
                y = y * 1000 if name == "tp" else y
                reg = reg[0]
                pvals_ = pvals.filter(pl.col("region") == reg)
                pvals_ = pvals_[name].to_numpy()
                ax.plot(
                    x[filter_],
                    y[filter_],
                    color=colors_regions[reg - 1],
                    label=DUNCANS_REGIONS_NAMES[reg - 1],
                    lw=1,
                )
                ax.plot(
                    np.where(pvals_[filter_] < 0.1, x[filter_], np.nan),
                    np.where(pvals_[filter_] < 0.1, y[filter_], np.nan),
                    color=colors_regions[reg - 1],
                    label=DUNCANS_REGIONS_NAMES[reg - 1],
                    lw=3,
                )
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1),
                xycoords="axes fraction",
                xytext=(+0.5, -0.5 - 6.5 * float(k == 0)),
                textcoords="offset fontsize",
                fontsize="medium",
                verticalalignment="top",
                fontfamily="serif",
                bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
            )
    # axes[0, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"] + " [K]")
    axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"] + " [mm]")
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")
    axes[0, 1].set_xlabel("Time around onset [day]")
    axes[1, 1].set_xlabel("Time around onset [day]")

clu = Clusterplot(1, 1, (-10, 40, 35, 72), row_height=5, fig=subfigs[1])
cmap = colormaps.pastel
ax = clu.axes[0]
unique_clusters = np.arange(1, 7)
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
clusters_da.assign_coords(lon=clusters_da.lon - clu.central_longitude).plot(
    ax=ax, cmap=cmap, norm=norm, add_colorbar=False, add_labels=False
)
for j in range(6):
    lo = (
        clusters_da.lon.where(clusters_da == (j + 1)).mean().item()
        - j
        - 3 * (j == 0)
        + 2 * (j == 2)
        + 3 * (j == 1)
        - (j == 4)
        - clu.central_longitude
    )
    la = clusters_da.lat.where(clusters_da == (j + 1)).mean().item() - (j == 5) * 2
    color = "black"
    ax.text(
        lo,
        la,
        DUNCANS_REGIONS_NAMES[j],
        ha="center",
        va="center",
        fontweight="bold",
        color=color,
        fontsize=12,
    )

ax.annotate(
    f"{ascii_lowercase[k + 1]})",
    xy=(float(k == 0), 1),
    xycoords="axes fraction",
    xytext=(+0.5 - 2 * float(k == 0), -0.5),
    textcoords="offset fontsize",
    fontsize="medium",
    verticalalignment="top",
    fontfamily="serif",
    bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
)
figure.savefig(f"{FIGURES}/Persistence/region_timeseries_30spells.pdf")

In [ ]:
colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
figure = plt.figure(figsize=(10, 4.5), constrained_layout=True)
subfigs = figure.subfigures(1, 2, wspace=0.00, width_ratios=[6.5, 3.5])
n_bootstraps = 1000
width = 0.3
x = (np.arange(1, 7)[:, None] + np.array([-width / 2, width / 2])[None, :]).ravel()
dalpha = 0.5
ds = 0.4
dv = -0.2
colors_regions_2 = rgb_to_hsv(colors_regions[:, :3])
offset = np.asarray([0, ds, dv])[None, :] * np.asarray([-1, 0])[:, None]
colors_regions_2 = colors_regions_2 + offset[:, None, :]
colors_regions_2 = colors_regions_2.transpose(1, 0, 2).reshape(12, 3)
colors_regions_2 = np.clip(colors_regions_2, 0.01, 1)
colors_regions_2 = hsv_to_rgb(colors_regions_2)
signif = 0.1
with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
    fig = subfigs[0]
    axes = fig.subplots(2, 2, sharex="col", sharey="row")
    axes = axes.T
    for i, spell_of in enumerate(["STJ", "EDJ"]):
        spells = spells_list[spell_of].drop("jet ID")
        spells = extend_spells(spells, time_before=datetime.timedelta(days=4))
        times = create_bootstrapped_times(spells, summer, n_bootstraps)
        for j, (name, df) in enumerate({"t2m": region_T, "tp": region_tp}.items()):
            ax = axes[i, j]
            if name == "tp":
                df = df.with_columns(pl.col(name) * 1000)
            region_ts_masked = spells.join(df, on="time").sort(
                "region", "spell", "relative_index"
            )
            # region_ts_masked = region_ts_masked.group_by(
            #     "region", "spell", "relative_index", maintain_order=True
            # ).agg(pl.col(name).mean())
            arrs = region_ts_masked.group_by("region", pl.col("relative_index") > 0).agg(pl.col(name)).sort("region", "relative_index")
            region_ts_masked = times.join(df, on="time")
            region_ts_masked = region_ts_masked.group_by(
                "sample_index", "region", pl.col("relative_index") > 0
            ).agg(pl.col(name).median()).sort(
                "sample_index", "region", "relative_index"
            )
            pvals = region_ts_masked.group_by(
                "region", pl.col("relative_index") > 0, maintain_order=True
            ).agg(pl.col(name).rank().last() / n_bootstraps)
            pvals = pvals.with_columns(
                pl.min_horizontal(pl.col(name), 1 - pl.col(name)) * 2
            )
            region_ts_masked = region_ts_masked.group_by(
                "region", pl.col("relative_index") > 0, maintain_order=True
            ).agg(pl.col(name).last())
            bplot = ax.boxplot(
                arrs[name], 
                positions=x, 
                widths=width,
                patch_artist=True,
                sym="",
            )
            for patch, color, pval in zip(bplot["boxes"], colors_regions_2, pvals[name]):
                patch.set_facecolor(color)
                if pval < signif:
                    patch.set_linewidth(2)
            for median in bplot["medians"]:
                median.set_color("black")
            for x_, mean, color_regions_2, pval in zip(x, region_ts_masked[name], colors_regions_2, pvals[name]):
                if pval < signif:
                    ax.scatter(x_, mean, zorder=20, color="red", marker="+", lw=2)
            k = 2 * j + i
            ax.annotate(
                f"{ascii_lowercase[k]})",
                xy=(0, 1),
                xycoords="axes fraction",
                xytext=(+0.5, -0.5),
                textcoords="offset fontsize",
                fontsize="medium",
                verticalalignment="top",
                fontfamily="serif",
                bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
            )
            ax.set_xticks(np.arange(1, 7), labels=DUNCANS_REGIONS_NAMES, rotation=35, ha="right")
    # axes[0, 0].legend(ncol=2, fontsize=13.5, labelspacing=0.3, markerscale=.1, columnspacing=.6, facecolor="white", framealpha=.5, fancybox=True, frameon=True)
    axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"] + " [K]")
    axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"] + " [mm]")
    axes[0, 0].set_title("STJ episodes")
    axes[1, 0].set_title("EDJ episodes")

clu = Clusterplot(1, 1, (-10, 40, 35, 72), row_height=5, fig=subfigs[1])
cmap = colormaps.pastel
ax = clu.axes[0]
unique_clusters = np.arange(1, 7)
norm = BoundaryNorm(np.arange(cmap.N) + 0.5, cmap.N)
clusters_da.assign_coords(lon=clusters_da.lon - clu.central_longitude).plot(
    ax=ax, cmap=cmap, norm=norm, add_colorbar=False, add_labels=False
)
for j in range(6):
    lo = (
        clusters_da.lon.where(clusters_da == (j + 1)).mean().item()
        - j
        - 3 * (j == 0)
        + 2 * (j == 2)
        + 3 * (j == 1)
        - (j == 4)
        - clu.central_longitude
    )
    la = clusters_da.lat.where(clusters_da == (j + 1)).mean().item() - (j == 5) * 2
    color = "black"
    ax.text(
        lo,
        la,
        DUNCANS_REGIONS_NAMES[j],
        ha="center",
        va="center",
        fontweight="bold",
        color=color,
        fontsize=12,
    )

ax.annotate(
    f"{ascii_lowercase[k + 1]})",
    xy=(float(k == 0), 1),
    xycoords="axes fraction",
    xytext=(+0.5 - 2 * float(k == 0), -0.5),
    textcoords="offset fontsize",
    fontsize="medium",
    verticalalignment="top",
    fontfamily="serif",
    bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
)
figure.savefig(f"{FIGURES}/Persistence/region_boxplot.pdf")

In [ ]:
# colors_regions = colormaps.pastel(np.linspace(0, 1, colormaps.pastel.N))[:6]
# timeseries = {}
# colors = {}
# for i, f in enumerate(Path(DATADIR, "ERA5/timeseries").glob("t*.parquet")):
#     timeseries[f.stem] = pl.read_parquet(f)
#     subname = f.stem.split("_")[0]
#     timeseries[f.stem] = (
#         timeseries[f.stem]
#         .rolling("time", period="1d", offset="-12h")
#         .agg(pl.col(subname).mean())
#     )
#     colors[f.stem] = colors_regions[i]
# n_bootstraps = 100
# with plt.style.context(["seaborn-v0_8-darkgrid", STYLE_SHEET]):
#     fig, axes = plt.subplots(2, 2, sharex="col", sharey="row")
#     axes = axes.T
#     for i, spell_of in enumerate(["STJ", "EDJ"]):
#         spells = spells_list[spell_of].drop("jet ID")
#         spells = extend_spells(spells, time_before=datetime.timedelta(days=4))
#         times = create_bootstrapped_times(spells, summer, n_bootstraps)
#         for name, df in timeseries.items():
#             subname, region = name.split("_")
#             j = int(subname == "tp")
#             ax = axes[i, j]
#             ts_masked = times.join(df, on="time").sort(
#                 "sample_index", "spell", "relative_index"
#             )
#             x, alive_spells = (
#                 ts_masked.filter(pl.col("sample_index") == n_bootstraps)
#                 .group_by("relative_index")
#                 .agg(pl.col("spell").n_unique())
#                 .sort("relative_index")
#                 .to_numpy()
#                 .T
#             )
#             season_mean = ts_masked[subname].mean()
#             ts_masked = ts_masked.group_by(
#                 "sample_index", "relative_index", maintain_order=True
#             ).agg(pl.col(subname).mean())
#             pvals = ts_masked.group_by("relative_index", maintain_order=True).agg(
#                 pl.col(subname).rank().last() / n_bootstraps
#             )
#             pvals = pvals.with_columns(
#                 pl.min_horizontal(pl.col(subname), 1 - pl.col(subname)) * 2
#             )
#             ts_masked = ts_masked.group_by("relative_index", maintain_order=True).agg(
#                 pl.col(subname).last()
#             )

#             x = x / 4
#             filter_ = alive_spells > 15
#             ax.axhline(0, color="black")
#             ax.axvline(0, color="black")
#             y = ts_masked[subname].to_numpy()
#             y = y * 1000 if subname == "tp" else y
#             season_mean = season_mean * 1000 if subname == "tp" else season_mean
#             pvals_ = pvals[subname].to_numpy()
#             ax.plot(
#                 x[filter_],
#                 y[filter_],
#                 color=colors[name],
#                 # label=region,
#                 lw=1,
#             )
#             if subname == "tp":
#                 ax.plot(
#                     x[filter_][[0, -1]],
#                     [season_mean, season_mean],
#                     color=colors[name],
#                     ls="dashed",
#                     lw=1,
#                 )
#             ax.plot(
#                 np.where(pvals_[filter_] < 0.1, x[filter_], np.nan),
#                 np.where(pvals_[filter_] < 0.1, y[filter_], np.nan),
#                 color=colors[name],
#                 label=region,
#                 lw=3,
#             )
#             k = 2 * j + i
#             ax.annotate(
#                 f"{ascii_lowercase[k]})",
#                 xy=(0, 1),
#                 xycoords="axes fraction",
#                 xytext=(+0.5, -0.5 - 6.5 * float(k > 0)),
#                 textcoords="offset fontsize",
#                 fontsize="medium",
#                 verticalalignment="top",
#                 fontfamily="serif",
#                 bbox=dict(facecolor="0.7", edgecolor="none", pad=3.0),
#             )
#     axes[1, 0].legend(
#         ncol=2,
#         fontsize=13.5,
#         labelspacing=0.3,
#         markerscale=0.1,
#         columnspacing=0.6,
#         facecolor="white",
#         framealpha=0.5,
#         fancybox=True,
#         frameon=True,
#     )
#     axes[0, 1].legend(
#         ncol=1,
#         fontsize=13.5,
#         labelspacing=0.3,
#         markerscale=0.1,
#         columnspacing=0.6,
#         facecolor="white",
#         framealpha=0.5,
#         fancybox=True,
#         frameon=True,
#     )

#     axes[0, 0].set_ylabel(PRETTIER_VARNAME["t2m"] + " [K]")
#     axes[0, 1].set_ylabel(PRETTIER_VARNAME["tp"] + " [mm]")
#     axes[0, 0].set_title("STJ episodes")
#     axes[1, 0].set_title("EDJ episodes")
#     axes[0, 1].set_xlabel("Time around onset [day]")
#     axes[1, 1].set_xlabel("Time around onset [day]")

In [ ]:
from matplotlib.colors import hex2color

names = ["Cyan", "Yellow", "Orange", "Purple", "Green", "Blue"]
names = [f"{name}Pastel" for name in names]
base = r"\definecolor{"
middle = r"}{rgb}{"
end = r"}"
for color, name in zip(cmap(norm(np.arange(1, 7))), names):
    color = [f"{spec:.3f}" for spec in color[:3]]
    print(base + f"{name}" + middle + " ".join(color) + end)

name = "STJcolor"
color = hex2color(COLORS[2])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

name = "EDJcolor"
color = hex2color(COLORS[1])
color = [f"{spec:.3f}" for spec in color[:3]]
print(base + f"{name}" + middle + " ".join(color) + end)

base = r"\textcolor{"
middle = r"}{\textbf{"
end = r"}}"
schema = []
for color, name in zip(names, DUNCANS_REGIONS_NAMES):
    schema.append(base + color + middle + name + end)
print(" & ".join(schema))

In [ ]:
for key, df in {"hw": hws, "pe": pes, "dr": drs}.items():
    out = summer.to_frame()

    for region in range(1, 7):
        alias = f"reg{region}"
        to_join = (
            df[["region", "time"]]
            .with_columns((pl.col("region") == region).alias(alias))
            .filter(alias)
            .drop("region")
        )
        out = out.join(to_join, on="time", how="left")
    for spell_of in ["STJ", "EDJ"]:
        spells = spells_list[f"{spell_of}"]
        spells = extend_spells(spells, time_before=datetime.timedelta(hours=6))
        spells = (
            spells[["time", "spell"]]
            .with_columns(spell=pl.lit(True))
            .rename({"spell": f"spell_{spell_of}"})
        )
        out = out.join(spells, on="time", how="left")
    out = out.fill_null(False).fill_nan(False)

    aggs = {}
    for i, spell_of in product(range(1, 7), ["STJ", "EDJ"]):
        aggs[f"{i}{spell_of}"] = odds_ratio(i, spell_of)
        aggs[f"{i}{spell_of}_signif"] = is_signif_OR(i, spell_of)
    out = out.select(**aggs)

    out = out.to_numpy().reshape(6, 4)
    overlaps_ = out[:, [0, 2]].T
    pvals_ = out[:, [1, 3]].T

    hoho = (
        pl.DataFrame(overlaps_, schema=DUNCANS_REGIONS_NAMES)
        .with_columns(
            [
                (pl.col(region)).round(1).cast(pl.String())
                for region in DUNCANS_REGIONS_NAMES
            ]
        )
        .with_columns(
            **{
                f"start{region}": pl.when(pl.lit(pl.Series(None, pvals_[:, i]) > 0.95))
                .then(pl.lit(r"$\mathbf{"))
                .otherwise(pl.lit(r"${"))
                for i, region in enumerate(DUNCANS_REGIONS_NAMES)
            }
        )
        .with_columns(
            **{
                region: pl.col(f"start{region}") + pl.col(region) + r"}$"
                for region in DUNCANS_REGIONS_NAMES
            }
        )
        .drop([f"start{region}" for region in DUNCANS_REGIONS_NAMES])
    )
    print(key, hoho)
    hoho.to_pandas().to_latex(
        buf=f"{RESULTS}/OR_{key}.tex",
        escape=False,
        column_format="l",
        multirow=False,
        header=True,
        index_names=False,
    )

# wrs

In [ ]:
wr_names = ["No", "GLBl", "ScTr", "EuBl", "ScBl"]
colors = ["#6C6C6C", "#7E7EF4", "#F2A685", "#8FC386", "green"]

grams_wr = pl.read_parquet(f"{DATADIR}/grams_wr.parquet")
grams_wr = grams_wr.with_columns(
    **{
        f"winner_{i}": pl.when(pl.col("winner") == i)
        .then(pl.lit(1.0))
        .otherwise(pl.lit(0.0))
        for i in range(5)
    }
)
width = 0.25
fig, axes = plt.subplot_mosaic(
    [["a)", "b)", "c)"], ["a)", "b)", "d)"]],
    figsize=(8, 4),
    constrained_layout=True,
    sharey=True,
    width_ratios=[1, 1, 0.5],
    gridspec_kw=dict(wspace=0.1),
)
for i, (spell_of, ax) in enumerate(zip(["STJ", "EDJ"], list(axes.values()))):
    grams_wr_masked = mask_from_spells_pl(
        spells_list[spell_of].drop("jet ID"),
        grams_wr,
        time_before=datetime.timedelta(days=3),
    )
    huh = grams_wr_masked.group_by(["relative_index"]).mean().sort("relative_index")
    alive_spells = (
        grams_wr_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    x = huh["relative_index"].to_numpy() / 4
    filter_ = alive_spells > 3
    x = x[filter_]
    bottom = np.zeros(len(x))
    for j in [*np.arange(1, 5), 0]:
        height = huh[f"winner_{j}"].to_numpy()[filter_]
        ax.bar(
            x,
            height,
            bottom=bottom,
            facecolor=colors[j],
            width=width,
            label=wr_names[j],
        )
        bottom = bottom + height
    ax.set_xlabel("Relative time around onset [day]")
    ax.set_title(
        f"{ascii_lowercase[i]}) {alive_spells.max():n} episodes of the {spell_of[:3]}"
    )
    ax.set_xlim(x[0] - width / 2, x[-1] + width / 2)
    ybounds = [1, 1.05]
    im = ax.pcolormesh(
        x,
        ybounds,
        alive_spells[filter_][None, 1:],
        zorder=2,
        cmap=colormaps.greys,
        alpha=0.7,
        vmin=0,
    )
    ax.axhline(1, color="black")
    ax.vlines(0, 0, 1, color="black", ls="dotted", lw=1.2)
    ax.grid(True, alpha=0.5, axis="y", color="black")
    ax.tick_params(length=0, axis="y")
h, l = axes["b)"].get_legend_handles_labels()
axes["c)"].set_axis_off()
axes["c)"].legend(h, l, fontsize=12, ncol=1, loc="upper left", title="WR names")
axes["a)"].set_ylabel("WR frequency")
ax = axes["d)"]
monthly_means = (
    grams_wr.filter(pl.col("time").dt.month() > 5)
    .group_by(pl.col("time").dt.month())
    .agg(*[pl.col(f"winner_{i}").mean() for i in range(5)])
    .sort("time")
)
x = np.array([6, 7, 8, 9])
bottom = np.zeros(len(x))
for j in [*np.arange(1, 5), 0]:
    height = monthly_means[f"winner_{j}"].to_numpy()
    ax.bar(x, height, bottom=bottom, facecolor=colors[j], width=0.9, label=wr_names[j])
    ax.grid(True, alpha=0.5, axis="y", color="black")
    bottom = bottom + height
ax.tick_params(length=0, axis="y")
ax.set_xticks(x, labels="JJAS")
ax.set_title("c) Monthly means")
fig.savefig(f"{FIGURES}/Persistence/wrs_bars.pdf")

# props

In [ ]:
from string import ascii_lowercase

data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_theta",
    "mean_lev",
    "mean_s",
    "width",
    "tilt",
    "waviness1",
    "waviness2",
    "wavinessDC16",
    "wavinessFV15",
    "mean_lat_var",
    "mean_s_var",
    "is_polar",
    "int",
    "int_over_europe",
]

for jet_name, spell in spells_list.items():
    fig = plot_relative_time(
        props_catd_summer,
        spell.drop("jet ID"),
        data_vars,
        1,
        4,
        4,
        row_height=2.3,
        col_width=3.1,
    )
    fig.savefig(f"{FIGURES}/Persistence/{jet_name}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase

data_vars = [
    "t2m-warm",
    "t2m-cold",
    "tp-warm",
    "tp-warm_entrance",
    "tp-warm_far_entrance",
    "tp-cold",
]

for jet_name, spell in spells_list.items():
    fig = plot_relative_time(
        average_jet_categories(props_with_extras_summer),
        spell.drop("jet ID"),
        data_vars,
        1,
        3,
        2,
        row_height=2.3,
        col_width=3.1,
        spaghetti=False,
    )
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase

data_vars = [
    "APVOany-warm",
    "APVOany-cold",
    "CPVOany-warm",
    "CPVOany-cold",
]

for jet_name, spell in spells_list.items():
    fig = plot_relative_time(
        average_jet_categories(props_with_extras_summer),
        spell.drop("jet ID"),
        data_vars,
        1,
        2,
        2,
        row_height=2.3,
        col_width=3.1,
    )
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase

data_vars = [
    "t300:grad-core",
    "t300:grad-core",
    "EKE250_5days-core",
    "mean_lat",
    "mean_s",
    "wavinessDC16",
]

for jet_name, spell in spells_list.items():
    fig = plot_relative_time(
        average_jet_categories(props_with_extras_summer),
        spell,
        data_vars,
        1,
        3,
        2,
        row_height=2.5,
        col_width=3.1,
    )
    # fig.savefig(f"{FIGURES}/Persistence/{jet}_props_30spells.pdf")
    # plt.close()

### reduced

In [ ]:
data_vars = [
    "mean_lat",
    "mean_s",
    "wavinessR16",
    "mean_s_var",
]
bigspells = pl.concat(list(spells_list.values()))
fig = plot_relative_time(
    props_catd_summer, bigspells, data_vars, 2, 2, 2, row_height=2.3, col_width=3.1, one_ax_each=True
)
fig.savefig(f"{FIGURES}/Persistence/both_props_30spells.pdf")

# jet length variability?

In [ ]:
from jetutils.geospatial import jet_integral_haversine
hoho = jets.group_by("time", "jet ID").agg(huh=jet_integral_haversine("lon", "lat", None, True), is_polar=pl.col("is_polar").mean())
hoho_summer_edjs = hoho.filter(pl.col("is_polar") > 0.6, pl.col("time").dt.month().is_in([6, 7, 8, 9]))
long_ones = hoho_summer_edjs.filter(pl.col("huh") > pl.col("huh").quantile(0.8))
short_ones = hoho_summer_edjs.filter(pl.col("huh") < pl.col("huh").quantile(0.2))
df = pl.scan_parquet(basepath.joinpath("tp_relative.parquet"))
to_plot_long = long_ones.lazy().join(df, on=["time", "jet ID"]).group_by("norm_index", "n").agg(pl.col("tp_interp").mean()).collect()
to_plot_short = short_ones.lazy().join(df, on=["time", "jet ID"]).group_by("norm_index", "n").agg(pl.col("tp_interp").mean()).collect()

In [ ]:
(polars_to_xarray(to_plot_short, ["n", "norm_index"]) * 1000).plot(vmin=0, vmax=2.0)
plt.gca().set_title("Short EDJs")
plt.figure()
(polars_to_xarray(to_plot_long, ["n", "norm_index"]) * 1000).plot(vmin=0, vmax=2.0)
plt.gca().set_title("Long EDJs")

In [ ]:
short_ones

In [ ]:

fig, ax = plt.subplots()
_, bins, _ = ax.hist(hoho.filter(pl.col("is_polar") < 0.4)["huh"].log10(), bins=51, density=True, label="Overall")
huhu = spells_list["STJ"].join(hoho.drop("is_polar"), on=["time", "jet ID"])
ax.hist(huhu["huh"].log10(), bins=bins, density=True, alpha=0.5, label="Persistent episodes")
ax.set_xlabel(r"$\log_{10}(l/(1\,\mathrm{m}))$")
ax.set_ylabel("Density")
ax.legend()
ax.set_title("STJ")

fig, ax1 = plt.subplots()
_, bins, _ = ax1.hist(hoho.filter(pl.col("is_polar") > 0.6)["huh"].log10(), bins=51, density=True, label="Overall")
huhu = spells_list["EDJ"].join(hoho.drop("is_polar"), on=["time", "jet ID"])
ax1.hist(huhu["huh"].log10(), bins=bins, density=True, alpha=0.5, label="Persistent episodes")
ax1.set_xlabel(r"$\log_{10}(l/(1\,\mathrm{m}))$")
ax1.set_ylabel("Density")
ax1.legend()
ax1.set_title("EDJ")
ax1.sharex(ax)


# realspace from data

In [ ]:
import polars_st as st


def centroid_jets(jets: pl.DataFrame):
    orig_jets = jets.clone()
    jets = jets.group_by("time", "jet").agg(
        points=pl.concat_arr(pl.col("lon"), pl.col("lat"))
    )
    points = st.linestring("points").st.set_srid(4326).st.to_srid(3857)
    points_right = st.linestring("points_right").st.set_srid(4326).st.to_srid(3857)
    frechet_expr = points.st.frechet_distance(points_right)
    cross = jets.join(jets, how="cross").select(
        "time", "jet", "time_right", "jet_right", frechet_expr
    )
    center_jet = (
        cross.group_by("time", "jet")
        .agg(pl.col("points").sum())
        .sort("points")
        .head(1)
        .join(orig_jets, on="time")
    )
    return center_jet


centroids_path = basepath.joinpath("centroids_during_spells.parquet")
if centroids_path.is_file():
    centroids = pl.read_parquet(centroids_path)
else:
    centroids = []
    for jet_name, spell_of in product(["STJ", "EDJ"], ["STJ", "EDJ"]):
        jets_ = (
            spells_list[spell_of]
            .select("time")
            .join(phat_jets, on="time")
            .filter(pl.col("jet") == jet_name)
            .with_columns(spell_of=pl.lit(spell_of))
        )
        centroids.append(centroid_jets(jets_))
    centroids = pl.concat(centroids)
    centroids.write_parquet(centroids_path)

## NATL and EUrope composites

In [ ]:
from jetutils.definitions import compute
from jetutils.stats import fdr_correction
from scipy.stats import rankdata

n_bootstraps = 100

def _func(da, times):
    result = []
    for spell in times["spell"].unique():
        huh_ = da.sel(
            time=polars_to_xarray(
                times.filter(pl.col("spell") == spell)[
                    "sample_index", "time", "relative_index"
                ],
                ["sample_index", "relative_index"],
            )
        )
        result.append(huh_)
    result = xr.concat(result, dim="spell", join="outer").mean(
        ["spell", "relative_index"]
    )
    result = result.copy(data=rankdata(result, axis=0) - 1)
    return result.isel(sample_index=-1).reset_coords("sample_index", drop=True) / (
        len(result.sample_index) - 1
    )


for var in ["t2m_anom", "tp_anom", "z500_anom"]:
    opath = Path(f"{FIGURES}/Persistence/realspace_mean/{var}.nc")
    if opath.is_file():
        continue
    this_ev = {}
    da = (
        xr.open_dataarray(f"{DATADIR}/ERA5/{var}.nc")
        .interp(time=summer, method="nearest")
        .load()
    )
    da = da.chunk({"time": -1, "lat": 15, "lon": 100})
    coords = {"lat": da.lat.values, "lon": da.lon.values}
    data = np.zeros([len(v) for v in coords.values()])
    template = xr.DataArray(data, coords=coords).chunk({"lat": 15, "lon": 100})
    for jet_name in ["STJ", "EDJ"]:
        key = f"{jet_name}_{var}"
        spells = spells_list[jet_name]
        times = create_bootstrapped_times(spells, summer, n_bootstraps)
        pvals = compute(
            da.map_blocks(_func, kwargs={"times": times}, template=template),
            progress_flag=True,
        )
        fdr_signif = pvals.copy(
            data=fdr_correction(pvals.values, 0.025, two_sided=True)
        )
        to_plot = da.sel(time=spells["time"]).mean(["time"])
        this_ev[key] = to_plot
        this_ev[key + "_pvals"] = pvals
        this_ev[key + "_fdr"] = fdr_signif
    xr.Dataset(this_ev).to_netcdf(opath)


for var, lev in product(["APVO_anom", "CPVO_anom"], [320, 330, 340, 350, "any"]):
    opath = Path(f"{FIGURES}/Persistence/realspace_mean/{var}{lev}.nc")
    if opath.is_file():
        continue
    this_ev = {}
    if lev != "any":
        da = (
            xr.open_dataarray(f"{DATADIR}/ERA5/{var}.nc")
            .sel(lev=lev)
            .reset_coords(["level", "lev"], drop=True)
            .interp(time=summer, method="nearest")
            .load()
        )
    else:
        da = (
            xr.open_dataarray(f"{DATADIR}/ERA5/{var}any.nc")
            .reset_coords(["level"], drop=True)
            .interp(time=summer, method="nearest")
            .load()
        )
    da = da.chunk({"time": -1, "lat": 15, "lon": 100})
    coords = {"lat": da.lat.values, "lon": da.lon.values}
    data = np.zeros([len(v) for v in coords.values()])
    template = xr.DataArray(data, coords=coords).chunk({"lat": 15, "lon": 100})
    for jet_name in ["STJ", "EDJ"]:
        key = f"{jet_name}_{var}"
        spells = spells_list[jet_name]
        times = create_bootstrapped_times(spells, summer, n_bootstraps)
        pvals = compute(
            da.map_blocks(_func, kwargs={"times": times}, template=template),
            progress_flag=True,
        )
        fdr_signif = pvals.copy(
            data=fdr_correction(pvals.values, 0.025, two_sided=True)
        )
        to_plot = da.sel(time=spells["time"]).mean(["time"])
        this_ev[key] = to_plot
        this_ev[key + "_pvals"] = pvals
        this_ev[key + "_fdr"] = fdr_signif
    xr.Dataset(this_ev).to_netcdf(opath)
# for var in ["APVO", "CPVO"]:
#     opath = Path(f"{FIGURES}/Persistence/realspace_mean/{var}.nc")
#     if opath.is_file():
#         continue
#     this_ev = {}
#     da = (
#         xr.open_dataarray(f"{DATADIR}/ERA5/{var}.nc")
#         .reset_coords("level", drop=True)
#         .interp(time=summer, method="nearest")
#         .astype(np.uint8)
#         .load()
#     )
#     da = da.chunk({"time": -1, "lev": 1, "lat": 15, "lon": 100})
#     coords = {"lev": da.lev.values, "lat": da.lat.values, "lon": da.lon.values}
#     data = np.zeros([len(v) for v in coords.values()])
#     template = xr.DataArray(data, coords=coords).chunk(
#         {"lev": 1, "lat": 15, "lon": 100}
#     )
#     for jet in ["STJ", "EDJ"]:
#         key = f"{jet}_{var}"
#         spells = spells_list[jet]
#         times = create_bootstrapped_times(spells, summer, n_bootstraps)
#         pvals = compute(
#             da.map_blocks(_func, kwargs={"times": times}, template=template),
#             progress_flag=True,
#         )
#         fdr_signif = pvals.copy(
#             data=fdr_correction(pvals.values, 0.025, two_sided=True)
#         )
#         to_plot = da.sel(time=spells["time"]).mean(["time"])
#         this_ev[key] = to_plot
#         this_ev[key + "_pvals"] = pvals
#         this_ev[key + "_fdr"] = fdr_signif
#     xr.Dataset(this_ev).to_netcdf(opath)

In [ ]:
from jetutils.data import smooth
for varname in ["APVO", "CPVO"]:
    da = xr.open_dataarray(f"{DATADIR}/ERA5/{varname}.nc").any("lev")
    clim = da.groupby("time.dayofyear").mean()
    clim = smooth(clim, {"dayofyear": ("win", 15)})
    clim.to_netcdf(f"{DATADIR}/ERA5/{varname}_anyclim.nc")
    anom = da.groupby("time.dayofyear") - clim
    anom.to_netcdf(f"{DATADIR}/ERA5/{varname}_anyanom.nc")

In [ ]:
region = (-80, 40, 0, 70)
region_2 = (-80, 40, 15, 80)
cbar_kwargs = {"shrink": 0.8, "fraction": 0.11, "pad": 0.03}
plot_kwargs = {
    "var": "t2m_anom",
    "cmap": colormaps.cmp_b2r,
    "levels": [-1.2, -0.8, -0.4, 0.4, 0.8, 1.2],
    "transparify": 1,
    "cbar_kwargs": cbar_kwargs | {"label": "2m temp. anom. [K]"},
}
stippling_kwargs = {
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xx",
}

VARNAME = plot_kwargs.pop("var")
for jet_name, region in zip(["STJ", "EDJ"], [region, region_2]):
    varname_ = VARNAME.replace("_anom", "")
    factor = FACTORS_UNITS.get(varname_, 1)
    clu = Clusterplot(1, 1, region, row_height=10)
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{VARNAME}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    to_plot = ds[f"{jet_name}_{VARNAME}"] * factor
    fdr = ds[f"{jet_name}_{VARNAME}_fdr"]
    clu.add_contourf(
        [to_plot],
        **plot_kwargs,
    )
    clu.add_stippling(fdr.expand_dims("jet"), **stippling_kwargs)
    
    varname = "tp_anom"
    varname_ = varname.replace("_anom", "")
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    fdr = ds[f"{jet_name}_{varname}_fdr"]
    # fdr = fdr.rolling({"lon": 3, "lat": 3}, min_periods=1).mean() > 0.2
    fdr = fdr * np.sign(ds[f"{jet_name}_{varname}"])
    clu.add_contour(
        [fdr],
        levels=[-0.5, 0.5],
        linewidths=2.0,
        colors=["lawngreen", "cyan"],
        linestyles=["solid", "solid"],
    )

    varname = "z500_anom"
    varname_ = varname.replace("_anom", "")
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    to_plot = ds[f"{jet_name}_{varname}"] / 9.8
    clu.add_contour([to_plot], levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False)
    centroids_ = centroids.filter(pl.col("spell_of") == jet_name)
    for (_, jetn), jet_ in centroids_.group_by("time", "jet"):
        if jetn == "EDJ":
            color = COLORS[1]
        else:
            color = COLORS[2]
        clu.axes[0].plot(
            jet_["lon"] - clu.central_longitude, jet_["lat"], color=color, lw=3
        )
    clu.fig.savefig(f"{FIGURES}/Persistence/composite_{jet_name}_NAtl.pdf")

In [ ]:
from jetutils.plots import make_transparent
from matplotlib.cm import ScalarMappable
from jetutils.data import extract_region

VARNAME = "tp_anom"
stippling_kwargs = dict(
    hatch="///",
    facecolor="none",
    edgecolor=COLORS_EXT[10],
    linewidth=0.0,
    hatch_linewidth=1,
)
jet_name = "STJ"
ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{VARNAME}.nc"))
# ds = ds.assign_coords(lon=ds.lon + 180)
da = ds[f"{jet_name}_tp_anom"] * 1000
fdr = ds[f"{jet_name}_tp_anom_fdr"] 
fdr = fdr.coarsen({"lon": 2, "lat": 2}, boundary="pad").mean() > 0.1
regions = {"A": (-80, 40, 0, 70), "B": (40, 280, -15, 15)}
transforms = {
    "A": ccrs.PlateCarree(central_longitude=0),
    "B": ccrs.PlateCarree(central_longitude=180),
}

hspace = 0.02
size_ratio = 0.06
cbar_size = 0.03
wspace = 0.02
levels = MaxNLocator(11, symmetric=True).tick_values(
    da.quantile(0.002), da.quantile(0.998)
)
# cmap = make_transparent(colormaps.BlWhRe, nlev=len(levels), direction=0, n_transparent=1)
cmap = colormaps.precip_diff_12lev
levels = np.delete(levels, len(levels) // 2)
norm = BoundaryNorm(levels, cmap.N, extend="both")
plot_kwargs = {"levels": levels, "cmap": cmap, "extend": "both"}
im = ScalarMappable(norm, cmap)
h1, w1 = regions["A"][1] - regions["A"][0], regions["A"][3] - regions["A"][2]
h2, w2 = regions["B"][1] - regions["B"][0], regions["B"][3] - regions["B"][2]
hr = h1 * w2 / h2 / w1
fig, axes = plt.subplot_mosaic(
    [["A", "edge"], ["B", "edge"]],
    height_ratios=(1, hr),
    width_ratios=(1, cbar_size),
    per_subplot_kw={key: dict(projection=val) for key, val in transforms.items()},
    figsize=(
        ((120 * size_ratio) + cbar_size) * (1 + hspace + 0.09),
        (70 * (1 + hr) * size_ratio) * (1 + wspace + 0.02),
    ),
    gridspec_kw=dict(wspace=wspace, hspace=hspace, bottom=0.01, top=0.99, left=0.01, right=0.99),
)

for key, region in regions.items():
    transform = transforms[key]
    axes[key].set_extent(region, crs=ccrs.PlateCarree())
    axes[key].set_ylim(*region[2:])
    axes[key].coastlines()
    if key == "B":
        da_ = da.assign_coords(lon=(da.lon.values) % 360).sortby("lon")
        fdr_ = fdr.assign_coords(lon=(fdr.lon.values) % 360).sortby("lon")
        stippling_kwargs_ = stippling_kwargs | {}
    else:
        da_ = da.copy()
        fdr_ = fdr.copy()
        stippling_kwargs_ = stippling_kwargs
    da_ = da_.sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    fdr_ = fdr_.sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    axes[key].contourf(da_.lon, da_.lat, da_, transform=ccrs.PlateCarree(), **plot_kwargs)
    c1 = axes[key].pcolor(
        fdr_.lon,
        fdr_.lat,
        fdr_.where(fdr_),
        transform=ccrs.PlateCarree(),
        **stippling_kwargs_,
    )
    
    bbox = axes[key].get_window_extent()
    
    # Convert display pixels to points (1 inch = 72 points)
    width_points = (bbox.width / fig.dpi) * 72
    height_points = (bbox.height / fig.dpi) * 72
    
    axes[key].annotate(
        f"{key.lower()})",
        (5, height_points - 5),
        xycoords="axes points",
        ha="left",
        va="top",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
    )
    
ax = axes["A"]
region = regions["A"]
varname = "t2m_anom"
varname_ = varname.replace("_anom", "")
ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
    lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
)
fdr = ds[f"{jet_name}_{varname}_fdr"]
# fdr = fdr.rolling({"lon": 3, "lat": 3}, min_periods=1).mean() > 0.2
fdr = fdr * np.sign(ds[f"{jet_name}_{varname}"])
ax.contour(
    fdr.lon,
    fdr.lat,
    fdr,
    levels=[-0.5, 0.5],
    linewidths=2.0,
    colors=["blue", "red"],
    linestyles=["solid", "solid"],
)

varname = "z500_anom"
varname_ = varname.replace("_anom", "")
ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
    lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
)
to_plot = ds[f"{jet_name}_{varname}"] / 9.8
ax.contour(
    to_plot.lon,
    to_plot.lat,
    to_plot,
    levels=[-60, -20, 20, 60],
    linewidths=3.0,
    colors="grey",
)
centroids_ = centroids.filter(pl.col("spell_of") == jet_name)
for (_, jetn), jet_ in centroids_.group_by("time", "jet"):
    if jetn == "EDJ":
        color = COLORS[1]
    else:
        color = COLORS[2]
    ax.plot(jet_["lon"] - clu.central_longitude, jet_["lat"], color=color, lw=3)
fig.colorbar(im, cax=axes["edge"], spacing="proportional", label="Precip. anomaly [mm]")
fig.savefig(f"{FIGURES}/Persistence/composite_{jet_name}.pdf")

In [ ]:
schema={"year": pl.UInt32(), "month": pl.UInt32(), "day": pl.UInt32(), "x": pl.Float32(), "y": pl.Float32(), "phase": pl.UInt8(), "A": pl.Float32()}

In [ ]:
region = (-180, 180, 0, 70)
cbar_kwargs = {"shrink": 0.8, "fraction": 0.11, "pad": 0.03}
plot_kwargs = {
    "var": "tp_anom",
    "cmap": colormaps.precip_diff_12lev,
    "levels": [-1.2, -0.8, -0.4, 0.4, 0.8, 1.2],
    "transparify": 1,
    "cbar_kwargs": cbar_kwargs | {"label": "2m temp. anom. [K]"},
}
stippling_kwargs = {
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xx",
}

VARNAME = plot_kwargs.pop("var")
for jet_name, region in zip(["STJ", "EDJ"], [region, region_2]):
    varname_ = VARNAME.replace("_anom", "")
    factor = FACTORS_UNITS.get(varname_, 1)
    clu = Clusterplot(1, 1, region, row_height=10)
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{VARNAME}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    to_plot = ds[f"{jet_name}_{VARNAME}"] * factor
    fdr = ds[f"{jet_name}_{VARNAME}_fdr"]
    clu.add_contourf(
        [to_plot],
        **plot_kwargs,
    )
    clu.add_stippling(fdr.expand_dims("jet"), **stippling_kwargs)
    varname = "tp_anom"
    varname_ = varname.replace("_anom", "")
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    fdr = ds[f"{jet_name}_{varname}_fdr"]
    # fdr = fdr.rolling({"lon": 3, "lat": 3}, min_periods=1).mean() > 0.2
    fdr = fdr * np.sign(ds[f"{jet_name}_{varname}"])
    clu.add_contour(
        [fdr],
        levels=[-0.5, 0.5],
        linewidths=2.0,
        colors=["lawngreen", "cyan"],
        linestyles=["solid", "solid"],
    )

    varname = "z500_anom"
    varname_ = varname.replace("_anom", "")
    ds = xr.open_dataset(Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    to_plot = ds[f"{jet_name}_{varname}"] / 9.8
    clu.add_contour([to_plot], levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False)
    centroids_ = centroids.filter(pl.col("spell_of") == jet_name)
    for (_, jetn), jet_ in centroids_.group_by("time", "jet"):
        if jetn == "EDJ":
            color = COLORS[1]
        else:
            color = COLORS[2]
        clu.axes[0].plot(
            jet_["lon"] - clu.central_longitude, jet_["lat"], color=color, lw=3
        )
    clu.fig.savefig(f"{FIGURES}/Persistence/composite_{jet_name}_NAtl.pdf")

In [ ]:
region = (-80, 40, 15, 80)

cbar_kwargs = {"shrink": 0.8, "fraction": 0.07, "pad": 0.01, "location": "bottom"}
plot_kwargs = {
    "var": "APVO_anom",
    "cmap": colormaps.bam,
    "levels": np.linspace(-16, 16, 17).tolist(),
    "transparify": 0,
    "cbar_kwargs": cbar_kwargs | {"label": "AWB freq anom. [p.p.]"},
}
stippling_kwargs = {
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xx",
}
jets_name = ["EDJ"]
nrow, ncol = 3, 2
colors_wb = ["mediumseagreen", "orangered"]
varname = plot_kwargs.pop("var")
varname_ = varname.replace("_anom", "")
clu = Clusterplot(nrow, ncol, region, row_height=4)
contourf_plot = []
contour_plot = []
fdr_plot = []
titles = []
for lev in [320, 330, 340, 350, "any"]:
    factor = FACTORS_UNITS.get(varname_, 1)
    ds = xr.open_dataset(
        Path(FIGURES, "Persistence/realspace_mean", f"{varname}{lev}.nc")
    ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    to_plot = ds[f"{jets_name[0]}_{varname}"] * factor
    fdr = ds[f"{jets_name[0]}_{varname}_fdr"]
    contourf_plot.append(to_plot)
    fdr_plot.append(fdr)
    varname2 = "CPVO_anom"
    varname2_ = varname2.replace("_anom", "")
    ds = xr.open_dataset(
        Path(FIGURES, "Persistence/realspace_mean", f"{varname2}{lev}.nc")
    ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    to_plot = ds[f"{jets_name[0]}_{varname2}"]
    fdr = ds[f"{jets_name[0]}_{varname2}_fdr"]
    coarsen = {"dim": {"lon": 4, "lat": 4}, "boundary": "trim"}
    fdr = (fdr.coarsen(**coarsen).mean() > .01) * np.sign(to_plot.coarsen(**coarsen).mean())
    contour_plot.append(fdr)
    titles.append(fr"$\theta={lev}\,\mathrm" + r"{K}$")
    
contourf_plot = xr.concat(contourf_plot, "lev")
fdr_plot = xr.concat(fdr_plot, "lev")
contour_plot = xr.concat(contour_plot, "lev")
clu.add_contourf(
    contourf_plot,
    **plot_kwargs,
    titles=titles,
)
clu.add_stippling(fdr_plot, **stippling_kwargs)
clu.add_contour(
    contour_plot,
    levels=[-.5, .5],
    linewidths=2.0,
    colors=["black", "black"],
    linestyles=["dashed", "solid"],
)
for ax in clu.axes:
    centroids_ = centroids.filter(pl.col("spell_of") == jets_name[0])
    for (_, jetn), jet_name in centroids_.group_by("time", "jet"):
        if jetn == "EDJ":
            color = COLORS[1]
        else:
            color = COLORS[2]
        ax.plot(
            jet_name["lon"] - clu.central_longitude, jet_name["lat"], color=color, lw=3
        )
clu.resize_relative([.8, 1.03])
clu.fig.savefig(f"{FIGURES}/Persistence/composites_EDJ_NAtl_RWB.pdf")

In [ ]:
region = (-80, 40, 15, 80)

cbar_kwargs = {"shrink": 0.8, "fraction": 0.07, "pad": 0.01, "location": "bottom"}
plot_kwargs = {
    "var": "APVO_anom",
    "cmap": colormaps.bam,
    "levels": np.linspace(-16, 16, 17).tolist(),
    "transparify": 0,
    "cbar_kwargs": cbar_kwargs | {"label": "AWB freq anom. [p.p.]"},
}
stippling_kwargs = {
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xx",
}
jets_name = ["STJ"]
nrow, ncol = 3, 2
colors_wb = ["mediumseagreen", "orangered"]
varname = plot_kwargs.pop("var")
varname_ = varname.replace("_anom", "")
clu = Clusterplot(nrow, ncol, region, row_height=4)
contourf_plot = []
contour_plot = []
fdr_plot = []
titles = []
for lev in [320, 330, 340, 350, "any"]:
    factor = FACTORS_UNITS.get(varname_, 1)
    ds = xr.open_dataset(
        Path(FIGURES, "Persistence/realspace_mean", f"{varname}{lev}.nc")
    ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    to_plot = ds[f"{jets_name[0]}_{varname}"] * factor
    fdr = ds[f"{jets_name[0]}_{varname}_fdr"]
    contourf_plot.append(to_plot)
    fdr_plot.append(fdr)
    varname2 = "CPVO_anom"
    varname2_ = varname2.replace("_anom", "")
    ds = xr.open_dataset(
        Path(FIGURES, "Persistence/realspace_mean", f"{varname2}{lev}.nc")
    ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    to_plot = ds[f"{jets_name[0]}_{varname2}"]
    fdr = ds[f"{jets_name[0]}_{varname2}_fdr"]
    coarsen = {"dim": {"lon": 4, "lat": 4}, "boundary": "trim"}
    fdr = (fdr.coarsen(**coarsen).mean() > .01) * np.sign(to_plot.coarsen(**coarsen).mean())
    contour_plot.append(fdr)
    titles.append(fr"$\theta={lev}\,\mathrm" + r"{K}$")
    
contourf_plot = xr.concat(contourf_plot, "lev")
fdr_plot = xr.concat(fdr_plot, "lev")
contour_plot = xr.concat(contour_plot, "lev")
clu.add_contourf(
    contourf_plot,
    **plot_kwargs,
    titles=titles,
)
clu.add_stippling(fdr_plot, **stippling_kwargs)
clu.add_contour(
    contour_plot,
    levels=[-.2, .2],
    linewidths=2.0,
    colors=["black", "black"],
    linestyles=["dashed", "solid"],
)
for ax in clu.axes:
    centroids_ = centroids.filter(pl.col("spell_of") == jets_name[0])
    for (_, jetn), jet_name in centroids_.group_by("time", "jet"):
        if jetn == "EDJ":
            color = COLORS[1]
        else:
            color = COLORS[2]
        ax.plot(
            jet_name["lon"] - clu.central_longitude, jet_name["lat"], color=color, lw=3
        )
clu.resize_relative([.8, 1.03])
clu.fig.savefig(f"{FIGURES}/Persistence/composites_STJ_NAtl_RWB.pdf")

In [ ]:
region = (-10, 40, 30, 70)
cbar_kwargs = {"shrink": 0.8, "fraction": 0.11, "pad": 0.03}
plot_kwargs_1 = {
    "var": "t2m_anom",
    "cmap": colormaps.cmp_b2r,
    "levels": [-1.2, -0.8, -0.4, 0.4, 0.8, 1.2],
    "transparify": 1,
    "cbar_kwargs": cbar_kwargs | {"label": "2m temp. anom. [K]"},
}
plot_kwargs_2 = {
    "var": "tp_anom",
    "cmap": colormaps.brbg,
    "levels": np.linspace(-.5, .5, 9).tolist(),
    "transparify": 0,
    "cbar_kwargs": cbar_kwargs | {"label": "Precip. anom. [mm]"},
}
stippling_kwargs = {
    "linewidth": 0.7,
    "color": "black",
    "hatch": "xx",
}
nrow, ncol = 1, 2
n = nrow * ncol
colors_wb = ["mediumseagreen", "orangered"]
bigfig = plt.figure(figsize=(6.8, 5.5), constrained_layout=True)
jets_name = ["STJ", "EDJ"]
subfigs = bigfig.subfigures(2, 1)
for plot_kwargs, fig in zip([plot_kwargs_1, plot_kwargs_2], subfigs):
    varname = plot_kwargs.pop("var")
    varname_ = varname.replace("_anom", "")
    ds = xr.open_dataset(
        Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")
    ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
    clu = Clusterplot(nrow, ncol, region, fig=fig)
    factor = FACTORS_UNITS.get(varname_, 1)
    titles = []
    for j, jet_name in enumerate(spells_list):
        letter = ascii_lowercase[
            j % len(ascii_lowercase) + len(clu.axes) * int(varname_ == "tp")
        ]
        n_spells = spells_list[jet_name]["spell"].n_unique()
        titles.append(f"{letter}) {jet_name[:3]}, {n_spells} episodes")
    to_plot = [ds[f"{jet}_{varname}"] * factor for jet in jets_name]
    fdr = xr.concat([ds[f"{jet}_{varname}_fdr"] for jet in jets_name], "jet")
    clu.add_contourf(
        to_plot,
        titles=titles,
        **plot_kwargs,
    )
    clu.add_stippling(fdr, **stippling_kwargs)
    if varname_ == "t2m":
        varname = "z500_anom"
        varname_ = varname.replace("_anom", "")
        ds = xr.open_dataset(
            Path(FIGURES, "Persistence/realspace_mean", f"{varname}.nc")
        ).sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3]))
        to_plot = [ds[f"{jet}_{varname}"] / 9.8 for jet in jets_name]
        clu.add_contour(
            to_plot, levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False
        )
    for j, jet_name in enumerate(spells_list):
        centroids_ = centroids.filter(pl.col("spell_of") == jet_name)
        for (_, jetn), jet in centroids_.group_by("time", "jet"):
            if jetn == "EDJ":
                color = COLORS[1]
            else:
                color = COLORS[2]
            clu.axes[j].plot(
                jet["lon"] - clu.central_longitude, jet["lat"], color=color, lw=3
            )
#     # clu.add_contour(
#     #     data_contour, levels=[-60, -20, 20, 60], linewidths=2.0, clabels=False
#     # )
#     fig.suptitle(PRETTIER_VARNAME.get(varname, varname))
bigfig.savefig(f"{FIGURES}/Persistence/composites_Europe.pdf")

## case studies

In [ ]:
from jetutils.anyspell import extend_spells_jet_ID
from jetutils.data import compute_anomalies_pl, coarsen_da

which_ts = [-1, 0, 3]
n_t = len(which_ts)
rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"//",
    "C": r"\\",
}
rwb_lw = 2
cbar_kwargs = {"location": "bottom", "shrink": 0.7, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "var": "t2m_anom",
        "cmap": colormaps.BlueWhiteOrangeRed,
        "levels": MaxNLocator(7).tick_values(-12, 12).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "var": "theta",
        "cmap": colormaps.bilbao_r,
        "levels": MaxNLocator(6).tick_values(300, 390).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
}
all_jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
for spell_of in ["STJ", "EDJ"]:
    is_edj = spell_of == "EDJ"
    region = (-80, 40, 15, 80) if is_edj else (-80, 40, 0, 65)
    plot_kwargs = plot_kwargs_all[spell_of]
    spells = spells_list[spell_of]
    spells = extend_spells_jet_ID(spells, props, time_before=datetime.timedelta(days=1))
    f1 = (pl.col("relative_time") / datetime.timedelta(days=1)).floor().is_in(which_ts)
    f2 = pl.col("time").dt.hour() == 12
    spells = spells.filter(f1, f2)
    var = plot_kwargs.pop("var")
    var_ = var.replace("_anom", "")
    da = xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    for n in range(30):
        ofile = Path(f"{FIGURES}/Persistence/case_studies/cs_{spell_of}_{n}.pdf")
        if ofile.is_file():
            continue
        clu = Clusterplot(n_t, 1, region, row_height=3.5)
        fig = clu.fig
        spell = spells.filter(pl.col("spell") == n)
        len_spell = spell["len"].first()
        times = spell["time"].unique().sort()[:n_t]
        titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
        titles = [
            f"{letter}) {title}, ${d=}$"
            for letter, title, d in zip(ascii_lowercase, titles, which_ts)
        ]
        to_plot = da.sel(time=times)
        clu.add_contourf(
            to_plot,
            titles=titles,
            cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
            **plot_kwargs,
        )
        fig.suptitle(f"{spell_of} lifecycle, " + str(len_spell / 4) + " days")
        for i, t in enumerate(times):
            jets_ = all_jets.filter(pl.col("time") == t)
            for jid, jet_name in jets_.group_by("jet ID"):
                jid = jid[0]
                lo, la, is_p = jet_name[["lon", "lat", "is_polar"]]
                if jid == spell["jet ID"][i]:
                    this_jet_is_p = spell_of == "EDJ"
                else:
                    this_jet_is_p = len(is_p) > 0 and is_p.mean() > 0.5
                color = COLORS[1] if this_jet_is_p else COLORS[2]
                lw = 2 + 2 * (jid == spell["jet ID"][i])
                clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)

        var = "tp_anom"
        var_ = var.replace("_anom", "")
        to_plot = xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
            lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
        )
        to_plot = coarsen_da(to_plot.sel(time=times), 3)
        color_tp = ["aquamarine"] if is_edj else ["cyan"]
        clu.add_contour(
            to_plot * 1000,
            levels=[5],
            colors=color_tp,
            linewidths=[1.4],
            linestyles=["solid"],
        )
        if spell_of == "EDJ":
            for orientation, hatch in rwb_hatching.items():
                var = f"{orientation}PVO"
                to_plot = (
                    xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc"))
                    .sel(
                        lev=340,
                        lon=slice(region[0], region[1]),
                        lat=slice(region[2], region[3]),
                    )
                    .reset_coords("level", drop=True)
                )
                to_plot = to_plot.sel(time=times)
                for ax, ouais in zip(clu.axes, to_plot):
                    cs = ax.pcolor(
                        to_plot.lon.values - clu.central_longitude,
                        to_plot.lat.values,
                        ouais.where(ouais),
                        hatch=hatch,
                        facecolor="none",
                        edgecolor=rwb_color["S"],
                        hatch_linewidth=rwb_lw,
                        linewidth=0,
                        zorder=100,
                    )
        else:
            var = "z500_anom"
            var_ = var.replace("_anom", "")
            to_plot = (
                xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
                    lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
                )
                / 9.8
            )
            to_plot = to_plot.sel(time=times)
            clu.add_contour(
                to_plot, levels=[-150, -75, 75, 150], colors="black", linewidths=2.0
            )
        fig.savefig(ofile)
        plt.close(fig)

In [ ]:
from jetutils.anyspell import extend_spells_jet_ID
from jetutils.data import compute_anomalies_pl, coarsen_da
plt.rcParams['path.simplify'] = True

figure = plt.figure(figsize=(7.2, 8), constrained_layout=True)
subfigs = figure.subfigures(1, 2)
which_ts = {
    "STJ": [0, 2, 4],
    "EDJ": [0, 4, 9],
}
n_t = len(which_ts["STJ"])
which = {"STJ": 20, "EDJ": 9}
rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"//",
    "C": r"\\",
}
rwb_lw = 1.5
cbar_kwargs = {"location": "bottom", "shrink": 0.7, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "var": "t2m_anom",
        "cmap": colormaps.BlueWhiteOrangeRed,
        "levels": MaxNLocator(7).tick_values(-15, 15).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "var": "theta",
        "cmap": colormaps.bilbao_r,
        "levels": [310, 320, 330, 340, 350, 360, 370],
        "cbar_kwargs": cbar_kwargs,
        # "var": "t2m_anom",
        # "cmap": colormaps.BlueWhiteOrangeRed,
        # "levels": MaxNLocator(7).tick_values(-12, 12).tolist(),
        # "cbar_kwargs": cbar_kwargs,
    },
}
all_jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
for i, spell_of in enumerate(["STJ", "EDJ"]):
    is_edj = spell_of == "EDJ"
    region = (-80, 40, 15, 80) if is_edj else (-80, 40, 0, 65)
    plot_kwargs = plot_kwargs_all[spell_of]
    spells = spells_list[spell_of]
    n = which[spell_of]
    print(spells.filter(pl.col("spell") == n)["time"].sort()[[0, -1]])
    spells = extend_spells_jet_ID(spells, props, time_before=datetime.timedelta(days=1))
    f1 = (
        (pl.col("relative_time") / datetime.timedelta(days=1))
        .floor()
        .is_in(which_ts[spell_of])
    )
    f2 = pl.col("time").dt.hour() == 12
    spells = spells.filter(f1, f2)
    var = plot_kwargs.pop("var")
    var_ = var.replace("_anom", "")
    da = xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    fig = subfigs[i]
    clu = Clusterplot(n_t, 1, region, row_height=3.5, fig=fig)
    spell = spells.filter(pl.col("spell") == n)
    len_spell = spell["len"].first()
    times = spell["time"].unique().sort()[:n_t]
    titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
    titles = [
        f"{letter}) {title}"
        for letter, title, d in zip(
            ascii_lowercase[n_t * i :], titles, which_ts[spell_of]
        )
    ]
    to_plot = da.sel(time=times)
    clu.add_contourf(
        to_plot,
        titles=titles,
        cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
        **plot_kwargs,
    )
    fig.suptitle(f"{spell_of} lifecycle, " + str(len_spell / 4) + " days")
    for i, t in enumerate(times):
        jets_ = all_jets.filter(pl.col("time") == t)
        for jid, jet_name in jets_.group_by("jet ID"):
            jid = jid[0]
            lo, la, is_p = jet_name[["lon", "lat", "is_polar"]]
            if jid == spell["jet ID"][i] and which_ts[spell_of][i] >= 0:
                this_jet_is_p = spell_of == "EDJ"
            else:
                this_jet_is_p = len(is_p) > 0 and is_p.mean() > 0.5
            color = COLORS[1] if this_jet_is_p else COLORS[2]
            lw = 2 + 2 * (jid == spell["jet ID"][i])
            clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)

    var = "tp_anom"
    var_ = var.replace("_anom", "")
    to_plot = xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
        lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
    )
    to_plot = coarsen_da(to_plot.sel(time=times), 3)
    color_tp = ["lawngreen", "aquamarine"] if is_edj else ["lawngreen", "cyan"]
    # color_tp = ["cyan"]
    clu.add_contour(
        to_plot * 1000,
        levels=[-5, 5],
        colors=color_tp,
        linewidths=[1.4],
        linestyles=["solid"],
    )
    if spell_of == "EDJ":
        for orientation, hatch in rwb_hatching.items():
            var = f"{orientation}PVO"
            to_plot = (
                xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc"))
                .sel(
                    lon=slice(region[0], region[1]),
                    lat=slice(region[2], region[3]),
                )
                .any("lev")
                .reset_coords("level", drop=True)
            )
            to_plot = to_plot.sel(time=times)
            for ax, ouais in zip(clu.axes, to_plot):
                cs = ax.pcolor(
                    to_plot.lon.values - clu.central_longitude,
                    to_plot.lat.values,
                    ouais.where(ouais),
                    hatch=hatch,
                    facecolor="none",
                    edgecolor=rwb_color["S"],
                    hatch_linewidth=rwb_lw,
                    linewidth=0,
                    zorder=100,
                    rasterized=True
                )
    else:
        var = "z500_anom"
        var_ = var.replace("_anom", "")
        to_plot = (
            xr.open_dataarray(Path(DATADIR, f"ERA5/{var}.nc")).sel(
                lon=slice(region[0], region[1]), lat=slice(region[2], region[3])
            )
            / 9.8
        )
        to_plot = to_plot.sel(time=times)
        clu.add_contour(
            to_plot, levels=[-150, -75, 75, 150], colors="black", linewidths=2.0
        )

figure.savefig(f"{FIGURES}/Persistence/case_studies/the_one.pdf")

# Demos

In [ ]:
name = "tp_anom"
da_precip_anom = xr.open_dataarray(f"{DATADIR}/ERA5/{name}.nc")
regions = {
    "tropical_atlantic": (-80, 40, -2, 15),
    "tropical_indian": (40, 160, -2, 15),
    "tropical_pacific": (160, -80, -2, 15),
}
for key, region in regions.items():
    if key == "tropical_pacific":
        ts = da_precip_anom.assign_coords(lon=da_precip_anom.lon % 360).sel(lon=slice(region[0] + 180, region[1] + 180), lat=slice(region[2], region[3])).mean(["lon", "lat"])
    else:
        ts = da_precip_anom.sel(lon=slice(region[0], region[1]), lat=slice(region[2], region[3])).mean(["lon", "lat"])
    ts = xarray_to_polars(ts)
    ts.write_parquet(f"{DATADIR}/ERA5/timeseries/{name}-{key}.parquet")
    # da = da_precip_anom.sel(lon=)

In [ ]:
import shapely
from jetutils.data import extract
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.LambertConformal(-20)})
gl = ax.gridlines(
    draw_labels=False, linewidth=0.5, alpha=0.4, color="k", linestyle="--"
)
region = (-82, 42, -2, 80)
ax.set_boundary(
    mpath.Path(
        [
            [region[0], region[2]],
            [region[1], region[2]],
            [region[1], region[3]],
            [region[0], region[3]],
        ],
        codes=[1, 2, 2, 5],
    ).interpolated(20),
    transform=ccrs.PlateCarree(),
)
ax.add_feature(cfeature.COASTLINE)
ax.set_extent(region)

boxes_precip = {
    "tropical": (-40, 39, 1, 14),
    "gulfstream": (-79, -50, 21, 45),
    "mid_atl": (-60, -20, 35, 55),
}
boxes_temp = {
    "tropical": (-80, 40, 0, 15),
    "west": (-80, -41, 20, 60),
    "east": (-39, -0, 20, 60),
}

ax.add_feature(cfeature.COASTLINE)

for name, box in boxes_precip.items():
    ax.add_geometries(
        shapely.box(*np.asarray(box)[[0, 2, 1, 3]]).segmentize(10),
        facecolor="none",
        edgecolor="dodgerblue",
        linewidth=2,
        crs=ccrs.PlateCarree(),
    )
for name, box in boxes_temp.items():
    ax.add_geometries(
        shapely.box(*np.asarray(box)[[0, 2, 1, 3]]).segmentize(10),
        facecolor="none",
        edgecolor="red",
        ls="dashed",
        linewidth=2,
        crs=ccrs.PlateCarree(),
    )
fig.savefig(f"{FIGURES}/Persistence/realspace_boxes_demo.pdf")

In [ ]:
from matplotlib.axes import Axes
from matplotlib.axis import Axis
from jetutils.plots import TEXTWIDTH_IN
from jetutils.geospatial import add_normals_meters, interp_jets_to_zero_one


def annotate(ax, annotation, coords=None):
    if coords is None:
        coords = (0.03, 0.97)
    ax.annotate(
        annotation,
        coords,
        xycoords="axes fraction",
        ha="left",
        va="top",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
    )
    return ax


cmap = LinearSegmentedColormap.from_list("", [COLORS[0], COLORS[0]], 20)
figure = plt.figure(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN + 1), constrained_layout=True)
subfigs = figure.subfigures(2, 1)
ax: Axis = subfigs[0].subplots(subplot_kw={"projection": ccrs.LambertConformal(-20)})
region = (-80, 40, 20, 75)
ax.set_boundary(
    mpath.Path(
        [
            [region[0], region[2]],
            [region[1], region[2]],
            [region[1], region[3]],
            [region[0], region[3]],
        ],
        codes=[1, 2, 2, 5],
    ).interpolated(20),
    transform=ccrs.PlateCarree(),
)
gl = ax.gridlines(
    draw_labels=False, linewidth=0.5, alpha=0.4, color="k", linestyle="--"
)
ax.add_feature(cfeature.COASTLINE)
ax.set_extent(region)
box = (-60, -40, 30, 40)
ax.add_geometries(
    shapely.box(*np.asarray(box)[[0, 2, 1, 3]]).segmentize(10),
    facecolor="none",
    edgecolor=cmap(0),
    linewidth=2,
    crs=ccrs.PlateCarree(),
)

jet_name = centroids.filter(pl.col("time") == pl.col("time").unique().get(0))
jet_with_normals = add_normals_meters(
    jet_name.filter(pl.col("index") % 2 == 0), half_length=2e6, dn=2e5
)
box = (
    pl.col("normallon") >= box[0],
    pl.col("normallon") <= box[1],
    pl.col("normallat") >= box[2],
    pl.col("normallat") <= box[3],
)
box = pl.when(box).then(1).otherwise(None)
jet_with_normals = jet_with_normals.with_columns(box=box)
jet_with_normals_normd = interp_jets_to_zero_one(
    jet_with_normals, ("box", "normallon", "normallat")
)

box2 = (0.5, 1, 0.0, 1)
filter_tau = (pl.col("index") >= pl.col("index").max() * box2[0]) & (
    pl.col("index") <= pl.col("index").max() * box2[1]
)
filter_n = (pl.col("n") / 1e6 >= box2[2]) & (pl.col("n") / 1e6 <= box2[3])
box_realspace_ = jet_with_normals.filter(filter_tau, filter_n)
a = box_realspace_.filter(pl.col("n") == 0).sort("index")
b = box_realspace_.filter(pl.col("index") == pl.col("index").max()).sort("n")
c = box_realspace_.filter(pl.col("n") == 1e6).sort("index", descending=True)
d = box_realspace_.filter(pl.col("index") == pl.col("index").min()).sort(
    "n", descending=True
)

box_realspace = np.concatenate(
    [side["normallon", "normallat"].to_numpy() for side in [a, b, c, d]]
)
box_realspace = shapely.simplify(shapely.Polygon(box_realspace), 0).segmentize(25)

ax.add_geometries(
    box_realspace,
    facecolor=(1, 1, 1, 0.95),
    edgecolor="black",
    linewidth=2,
    crs=ccrs.PlateCarree(),
    zorder=4,
)
ax.annotate(
    "Cold, exit",
    (box_realspace_["normallon"].mean() - 4, box_realspace_["normallat"].mean() + 2),
    ha="center",
    va="center",
    zorder=4,
    transform=ccrs.PlateCarree(),
    rotation=10,
)

for _, nmj in jet_with_normals.group_by("index"):
    ax.plot(
        nmj["normallon"],
        nmj["normallat"],
        transform=ccrs.PlateCarree(),
        color=COLORS[3],
        lw=2,
        zorder=2,
    )
ax.scatter(
    jet_with_normals["normallon"],
    jet_with_normals["normallat"],
    transform=ccrs.PlateCarree(),
    c=jet_with_normals["box"],
    cmap=cmap,
    marker="o",
    s=20,
    zorder=3,
)
ax.scatter(
    jet_name["lon"],
    jet_name["lat"],
    transform=ccrs.PlateCarree(),
    color=COLORS[1],
    s=20,
    zorder=5,
)
# ax.fill(
#     box_realspace_["normallon"][[0, 0, -1, -1, 0]],
#     box_realspace_["normallat"][[0, -1, -1, 0, 0]],
#     facecolor=(1, 1, 1, 0.95),
#     edgecolor="black",
#     linewidth=2,
#     transform=ccrs.PlateCarree(),
#     zorder=400
# )
ax = annotate(ax, "a")

axes = subfigs[1].subplots(1, 2, sharey=True)
ax = axes[0]
X = jet_with_normals["index"]
Y = jet_with_normals["n"] / 1e6
C = jet_with_normals["box"]
for _, nmj in jet_with_normals.group_by("index", maintain_order=True):
    nmj = nmj.sort("n")
    ax.plot(nmj["index"], nmj["n"] / 1e6, color=COLORS[3], lw=3)
ax.scatter(X, Y, c=C, zorder=200, cmap=cmap)
ax.axhline(0.0, color=COLORS[1], lw=2)
ax.set_xlabel("Index along jet")
ax.set_ylabel("Normal distance [1000 km]")
ax = annotate(ax, "b")

ax = axes[1]
ax: Axes = ax
X = jet_with_normals_normd["norm_index"]
Y = jet_with_normals_normd["n"] / 1e6
C = jet_with_normals_normd["box"]
for _, nmj in jet_with_normals_normd.group_by("norm_index", maintain_order=True):
    nmj = nmj.sort("n")
    ax.plot(nmj["norm_index"], nmj["n"] / 1e6, color=COLORS[3], lw=3)
ax.scatter(X, Y, c=C, zorder=200, cmap=cmap)
ax.axhline(0.0, color=COLORS[1], lw=2)
ax.set_xlabel("Normd. index along jet")
ax: Axes = annotate(ax, "c")
ax.add_patch(
    mpatches.Rectangle(
        (box2[0] + 0.02, box2[2] + 0.02),
        box2[1] - box2[0] - 0.02,
        box2[3] - box2[2] - 0.02,
        zorder=3,
        edgecolor="black",
        facecolor=(1, 1, 1, 0.95),
        linewidth=2,
    ),
)
ax.text(
    (box2[0] + box2[1]) / 2,
    (box2[2] + box2[3]) / 2,
    "Cold, exit",
    ha="center",
    va="center",
    zorder=4,
)

figure.savefig(f"{FIGURES}/FeatureBased/jetcentred_demo.pdf")